## 00_Config_Utilidades

<!-- merged from: 00_Config_Utilidades.ipynb (Jupy Tools) -->

# 00 · Configuración y utilidades compartidas
**Proyecto:** Modelo de aprendizaje automático para la predicción de la tasa de deserción escolar departamental en Colombia (2018–2024) basado en indicadores de salud pública.

Este notebook **no se ejecuta solo**: los notebooks 01–05 lo cargan con `%run ./00_Config_Utilidades`.
Centraliza en un único lugar (una sola fuente de verdad):

| Bloque | Contenido |
|---|---|
| Parámetros | catálogo, esquemas, rutas, periodo de estudio, ventanas de validación |
| Contratos de datos | columnas esperadas por fuente, alias aceptados, columnas obligatorias |
| Dimensión territorial | 33 departamentos con **código DIVIPOLA** como llave primaria (el nombre es solo para visualización) |
| Funciones de limpieza | normalización de texto y nombres, *parsing* numérico robusto (ANSI-safe), clasificación de eventos SIVIGILA |
| Gobierno de datos | registro de reglas de calidad (bloqueantes / advertencias) persistido en `gobierno_datos.control_calidad` |
| E/S | lectura/escritura de tablas Delta con nombres de tres niveles de Unity Catalog |

In [ ]:
import os
import re
import json
import uuid
import unicodedata
from datetime import datetime, timezone

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# ------------------------------------------------------------------
# Entorno de ejecución
# ------------------------------------------------------------------
# En Databricks existe la variable DATABRICKS_RUNTIME_VERSION. Fuera de
# Databricks (pruebas locales con PySpark) se usa Parquet y nombres de
# dos niveles, sin cambiar la lógica del pipeline.
EN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
FORMATO_TABLA = "delta" if EN_DATABRICKS else "parquet"

# ------------------------------------------------------------------
# Parámetros del proyecto
# ------------------------------------------------------------------
CATALOG = "desercion_escolar"
SCHEMA_BRONZE = "capa_bronce"
SCHEMA_SILVER = "capa_plata"
SCHEMA_GOLD = "capa_oro"
SCHEMA_GOB = "gobierno_datos"          # control de calidad, cargas y linaje

INPUT_PATH = globals().get(
    "INPUT_PATH_OVERRIDE", "/Volumes/desercion_escolar/capa_bronce/archivos_origen"
)
MODELOS_PATH = globals().get(
    "MODELOS_PATH_OVERRIDE", "/Volumes/desercion_escolar/capa_oro/modelos"
)
EXPORT_PATH = globals().get(
    "EXPORT_PATH_OVERRIDE", "/Volumes/desercion_escolar/capa_oro/exportes_powerbi"
)

# Periodo de estudio (unidad de análisis departamento-año)
ANIO_MIN = 2018
ANIO_MAX = 2024
# Educación (MEN) trae 2011-2024: se conserva desde 2011 en Silver para poder
# construir rezagos (t-1, t-2, t-3) del año 2018 sin imputar.
ANIO_MIN_EDUCACION = 2011
# Terridata: se conserva historia previa para la unión "as-of" (último dato
# publicado hasta t-1).
ANIO_MIN_POBREZA = 2005
# Año objetivo del pronóstico (escenario t+1)
ANIO_PRONOSTICO = ANIO_MAX + 1

# Protocolo de evaluación temporal (backtesting con ventana expansiva)
ANIOS_VALIDACION = [2020, 2021, 2022, 2023]   # selección de modelo
ANIO_TEST = 2024                               # se usa UNA sola vez
SEMILLA = 42

# ------------------------------------------------------------------
# Identificador de ejecución (trazabilidad entre notebooks de un job)
# ------------------------------------------------------------------
def _widget(nombre, defecto=""):
    try:
        dbutils.widgets.text(nombre, defecto)
        valor = dbutils.widgets.get(nombre)
        return valor if valor != "" else defecto
    except Exception:
        return defecto

EJECUCION_ID = _widget("ejecucion_id", "") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
print(f"Entorno Databricks: {EN_DATABRICKS} | formato: {FORMATO_TABLA} | ejecución: {EJECUCION_ID}")

Entorno Databricks: True | formato: delta | ejecución: 20261005T153505-6161da


## Entrada/salida de tablas

In [ ]:
def fqtn(schema: str, tabla: str) -> str:
    """Nombre completo de la tabla. Unity Catalog: catalogo.esquema.tabla."""
    return f"{CATALOG}.{schema}.{tabla}" if EN_DATABRICKS else f"{schema}.{tabla}"


def crear_esquemas():
    if EN_DATABRICKS:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    for s in [SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_GOB]:
        nombre = f"{CATALOG}.{s}" if EN_DATABRICKS else s
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {nombre}")
    if EN_DATABRICKS:
        # Volúmenes para modelos serializados y exportes (Power BI sin conector)
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}.modelos")
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}.exportes_powerbi")


def cachear(df: DataFrame) -> DataFrame:
    """cache() cuando el cómputo lo permite (en Databricks Serverless no está soportado: se omite)."""
    try:
        return df.cache()
    except Exception:
        return df


def liberar(df: DataFrame):
    try:
        df.unpersist()
    except Exception:
        pass


def existe_tabla(schema: str, tabla: str) -> bool:
    try:
        return spark.catalog.tableExists(fqtn(schema, tabla))
    except Exception:
        try:
            spark.table(fqtn(schema, tabla)).limit(0).collect()
            return True
        except Exception:
            return False


def leer_tabla(schema: str, tabla: str) -> DataFrame:
    return spark.table(fqtn(schema, tabla))


def guardar_tabla(df: DataFrame, schema: str, tabla: str, modo: str = "overwrite",
                  particion=None, comentario: str = None) -> int:
    """Escribe una tabla gestionada (Delta en Databricks) y devuelve el número de filas."""
    escritor = df.write.format(FORMATO_TABLA).mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if modo == "append" and EN_DATABRICKS:
        escritor = escritor.option("mergeSchema", "true")
    if particion:
        escritor = escritor.partitionBy(*([particion] if isinstance(particion, str) else particion))
    escritor.saveAsTable(fqtn(schema, tabla))
    if comentario and EN_DATABRICKS:
        spark.sql(f"COMMENT ON TABLE {fqtn(schema, tabla)} IS '{comentario.replace(chr(39), '')}'")
    n = spark.table(fqtn(schema, tabla)).count()
    print(f"  ✔ {fqtn(schema, tabla)} ({modo}) → {n:,} filas")
    return n

def a_spark(pdf_, esquema=None):
    """pandas → Spark con tipos Python nativos según el esquema (robusto con o sin Arrow)."""
    import numpy as np
    import pandas as pd
    if esquema is None:
        campos = []
        for c, dt in pdf_.dtypes.items():
            if pd.api.types.is_bool_dtype(dt):
                tipo = T.BooleanType()
            elif pd.api.types.is_integer_dtype(dt):
                tipo = T.LongType()
            elif pd.api.types.is_float_dtype(dt):
                tipo = T.DoubleType()
            else:
                tipo = T.StringType()
            campos.append(T.StructField(str(c), tipo, True))
        esquema = T.StructType(campos)
    if isinstance(esquema, str):
        esquema = spark.createDataFrame([], esquema).schema
    conv = []
    for campo in esquema.fields:
        t = campo.dataType
        if isinstance(t, (T.IntegerType, T.LongType)):
            conv.append(lambda v: int(v))
        elif isinstance(t, (T.DoubleType, T.FloatType)):
            conv.append(lambda v: float(v))
        elif isinstance(t, T.BooleanType):
            conv.append(lambda v: bool(v))
        elif isinstance(t, T.StringType):
            conv.append(lambda v: str(v))
        else:
            conv.append(lambda v: v)
    filas = []
    for fila in pdf_.itertuples(index=False, name=None):
        filas.append(tuple(None if (v is None or (isinstance(v, float) and np.isnan(v)) or v is pd.NA or v is pd.NaT)
                           else f(v.item() if hasattr(v, "item") else v) for v, f in zip(fila, conv)))
    return spark.createDataFrame(filas, esquema)

## Gobierno de datos: registro de reglas de calidad
Cada regla queda persistida con su valor observado, umbral, severidad y estado.
Las reglas **BLOQUEANTES** que fallan detienen el pipeline (después de persistir la evidencia), corrigiendo el hallazgo
*"validación no bloqueante"* del informe.

In [ ]:
_REGISTRO_CALIDAD = []

ESQUEMA_CALIDAD = T.StructType([
    T.StructField("ejecucion_id", T.StringType()),
    T.StructField("fecha_evaluacion", T.TimestampType()),
    T.StructField("capa", T.StringType()),
    T.StructField("tabla", T.StringType()),
    T.StructField("regla", T.StringType()),
    T.StructField("dimension_calidad", T.StringType()),
    T.StructField("severidad", T.StringType()),
    T.StructField("valor_observado", T.DoubleType()),
    T.StructField("umbral", T.StringType()),
    T.StructField("estado", T.StringType()),
    T.StructField("detalle", T.StringType()),
])


def verificar(capa, tabla, regla, cumple, valor=None, umbral=None,
              severidad="ADVERTENCIA", dimension="validez", detalle=""):
    """Registra una regla de calidad. Si es BLOQUEANTE y no se cumple, persiste y detiene."""
    cumple = bool(cumple)
    estado = "OK" if cumple else ("FALLA" if severidad == "BLOQUEANTE" else "ALERTA")
    _REGISTRO_CALIDAD.append((
        EJECUCION_ID, datetime.now(), capa, tabla, regla, dimension, severidad,
        None if valor is None else float(valor), None if umbral is None else str(umbral),
        estado, str(detalle)[:2000],
    ))
    icono = {"OK": "✅", "ALERTA": "⚠️", "FALLA": "❌"}[estado]
    print(f"{icono} [{capa}/{tabla}] {regla}: valor={valor} umbral={umbral} {detalle}")
    if estado == "FALLA":
        persistir_calidad()
        raise ValueError(f"Regla bloqueante incumplida en {capa}/{tabla}: {regla} (valor={valor}, umbral={umbral}). {detalle}")
    return cumple


def persistir_calidad():
    global _REGISTRO_CALIDAD
    if not _REGISTRO_CALIDAD:
        return
    df = spark.createDataFrame(_REGISTRO_CALIDAD, ESQUEMA_CALIDAD)
    guardar_tabla(df, SCHEMA_GOB, "control_calidad", modo="append")
    _REGISTRO_CALIDAD = []


def perfil_nulos(df: DataFrame, columnas=None) -> dict:
    """Porcentaje de nulos por columna (una sola pasada)."""
    columnas = columnas or [c for c in df.columns]
    total = df.count()
    if total == 0:
        return {c: 1.0 for c in columnas}
    fila = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in columnas]).first()
    return {c: (fila[c] or 0) / total for c in columnas}

## Normalización de texto y nombres

In [ ]:
def normalizar_nombre(nombre: str) -> str:
    """'Código Departamento' → 'codigo_departamento'. Sin tildes, minúsculas, solo [a-z0-9_]."""
    if nombre is None:
        return None
    nombre = unicodedata.normalize("NFKD", str(nombre))
    nombre = "".join(c for c in nombre if not unicodedata.combining(c))
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre.lower()).strip("_")
    return nombre or "columna_sin_nombre"


# UDF determinística para *slugs* de indicadores (catálogo pequeño, no afecta rendimiento)
slug_udf = F.udf(lambda s: normalizar_nombre(s)[:120] if s else None, T.StringType())

_TILDES_ORIGEN = "ÁÉÍÓÚÜÑÀÈÌÒÙÂÊÎÔÛÄËÏÖáéíóúüñàèìòù"
_TILDES_DESTINO = "AEIOUUNAEIOUAEIOUAEIOaeiouunaeiou"


def texto_normalizado(col):
    """Mayúsculas, sin tildes, sin puntuación y con espacios simples (expresión Spark)."""
    c = col if not isinstance(col, str) else F.col(col)
    c = F.translate(F.upper(F.trim(c.cast("string"))), _TILDES_ORIGEN, _TILDES_DESTINO.upper())
    c = F.regexp_replace(c, r"[.,;:()\-_/]", " ")
    c = F.trim(F.regexp_replace(c, r"\s+", " "))
    return F.when(c == "", None).otherwise(c)

## *Parsing* numérico robusto
Corrige el hallazgo *"conversión numérica riesgosa"*: la regla depende del **separador decimal declarado por fuente**
(contrato de formato) y nunca se usa `cast` directo (que en modo ANSI aborta la ejecución). Los valores no parseables quedan nulos
**y se cuentan** como regla de calidad.

| Fuente | Ejemplo crudo | Regla | Resultado |
|---|---|---|---|
| Terridata | `1.234.567,89` | decimal `,` → quitar `.` y cambiar `,` por `.` | 1234567.89 |
| Terridata | `49,70` | idem | 49.70 |
| MEN | `94,01%` | auto: si hay `,` es decimal | 94.01 |
| MEN | `74.48` | auto: sin `,` el `.` es decimal | 74.48 |

In [ ]:
_REGEX_NUMERO = r"^-?[0-9]+(\.[0-9]+)?([eE][-+]?[0-9]+)?$"
_VALORES_NULOS = ["", "NA", "N/A", "ND", "N.D.", "-", "--", "NULL", "NAN", "SIN DATO", "S.D."]


def parsear_numero(col, decimal: str = "auto"):
    """Convierte texto a double de forma segura. decimal: ',' | '.' | 'auto'."""
    c = col if not isinstance(col, str) else F.col(col)
    s = F.regexp_replace(c.cast("string"), r"[%\s $]", "")
    s = F.when(F.upper(s).isin(_VALORES_NULOS), None).otherwise(s)
    if decimal == ",":
        s = F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", ".")
    elif decimal == ".":
        s = F.regexp_replace(s, ",", "")
    else:
        s = F.when(s.contains(","), F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", ".")).otherwise(s)
    return F.when(s.rlike(_REGEX_NUMERO), s.cast("double")).otherwise(F.lit(None).cast("double"))


def parsear_entero(col):
    c = col if not isinstance(col, str) else F.col(col)
    s = F.regexp_replace(c.cast("string"), r"\s", "")
    return F.when(s.rlike(r"^-?[0-9]+(\.0+)?$"), s.cast("double").cast("int")).otherwise(F.lit(None).cast("int"))


def contar_fallas_parseo(df: DataFrame, pares) -> dict:
    """pares: lista (col_cruda, col_parseada). Cuenta valores no nulos en crudo que quedaron nulos."""
    exprs = [F.sum((F.col(c).isNotNull() & (F.trim(F.col(c).cast("string")) != "")
                    & ~F.upper(F.trim(F.col(c).cast("string"))).isin(_VALORES_NULOS)
                    & F.col(p).isNull()).cast("int")).alias(p) for c, p in pares]
    fila = df.select(exprs).first()
    return {p: int(fila[p] or 0) for _, p in pares}

## Dimensión territorial (llave DIVIPOLA)
Corrige el hallazgo *"homologación por nombre"*: el **código DIVIPOLA** de 2 dígitos es la llave de integración de las tres fuentes.
El nombre se usa solo para (i) validar consistencia código↔nombre y (ii) visualización. Las coordenadas son las de la capital
departamental (útiles para mapas de burbujas en Power BI).

In [ ]:
DEPARTAMENTOS = [
    # cod, nombre_canonico, nombre_visual, region, capital, lat, lon, codigo_iso
    (5,  "ANTIOQUIA", "Antioquia", "Andina", "Medellín", 6.2442, -75.5812, "CO-ANT"),
    (8,  "ATLANTICO", "Atlántico", "Caribe", "Barranquilla", 10.9685, -74.7813, "CO-ATL"),
    (11, "BOGOTA D C", "Bogotá D.C.", "Andina", "Bogotá", 4.7110, -74.0721, "CO-DC"),
    (13, "BOLIVAR", "Bolívar", "Caribe", "Cartagena", 10.3910, -75.4794, "CO-BOL"),
    (15, "BOYACA", "Boyacá", "Andina", "Tunja", 5.5353, -73.3678, "CO-BOY"),
    (17, "CALDAS", "Caldas", "Andina", "Manizales", 5.0703, -75.5138, "CO-CAL"),
    (18, "CAQUETA", "Caquetá", "Amazonía", "Florencia", 1.6144, -75.6062, "CO-CAQ"),
    (19, "CAUCA", "Cauca", "Pacífica", "Popayán", 2.4448, -76.6147, "CO-CAU"),
    (20, "CESAR", "Cesar", "Caribe", "Valledupar", 10.4631, -73.2532, "CO-CES"),
    (23, "CORDOBA", "Córdoba", "Caribe", "Montería", 8.7479, -75.8814, "CO-COR"),
    (25, "CUNDINAMARCA", "Cundinamarca", "Andina", "Bogotá", 5.0260, -74.0300, "CO-CUN"),
    (27, "CHOCO", "Chocó", "Pacífica", "Quibdó", 5.6947, -76.6611, "CO-CHO"),
    (41, "HUILA", "Huila", "Andina", "Neiva", 2.9273, -75.2819, "CO-HUI"),
    (44, "LA GUAJIRA", "La Guajira", "Caribe", "Riohacha", 11.5444, -72.9072, "CO-LAG"),
    (47, "MAGDALENA", "Magdalena", "Caribe", "Santa Marta", 11.2408, -74.1990, "CO-MAG"),
    (50, "META", "Meta", "Orinoquía", "Villavicencio", 4.1420, -73.6266, "CO-MET"),
    (52, "NARINO", "Nariño", "Pacífica", "Pasto", 1.2136, -77.2811, "CO-NAR"),
    (54, "NORTE DE SANTANDER", "Norte de Santander", "Andina", "Cúcuta", 7.8939, -72.5078, "CO-NSA"),
    (63, "QUINDIO", "Quindío", "Andina", "Armenia", 4.5339, -75.6811, "CO-QUI"),
    (66, "RISARALDA", "Risaralda", "Andina", "Pereira", 4.8133, -75.6961, "CO-RIS"),
    (68, "SANTANDER", "Santander", "Andina", "Bucaramanga", 7.1193, -73.1227, "CO-SAN"),
    (70, "SUCRE", "Sucre", "Caribe", "Sincelejo", 9.3047, -75.3978, "CO-SUC"),
    (73, "TOLIMA", "Tolima", "Andina", "Ibagué", 4.4389, -75.2322, "CO-TOL"),
    (76, "VALLE DEL CAUCA", "Valle del Cauca", "Pacífica", "Cali", 3.4516, -76.5320, "CO-VAC"),
    (81, "ARAUCA", "Arauca", "Orinoquía", "Arauca", 7.0847, -70.7591, "CO-ARA"),
    (85, "CASANARE", "Casanare", "Orinoquía", "Yopal", 5.3378, -72.3959, "CO-CAS"),
    (86, "PUTUMAYO", "Putumayo", "Amazonía", "Mocoa", 1.1522, -76.6464, "CO-PUT"),
    (88, "SAN ANDRES", "San Andrés y Providencia", "Insular", "San Andrés", 12.5847, -81.7006, "CO-SAP"),
    (91, "AMAZONAS", "Amazonas", "Amazonía", "Leticia", -4.2153, -69.9406, "CO-AMA"),
    (94, "GUAINIA", "Guainía", "Amazonía", "Inírida", 3.8653, -67.9239, "CO-GUA"),
    (95, "GUAVIARE", "Guaviare", "Amazonía", "San José del Guaviare", 2.5729, -72.6459, "CO-GUV"),
    (97, "VAUPES", "Vaupés", "Amazonía", "Mitú", 1.2536, -70.2345, "CO-VAU"),
    (99, "VICHADA", "Vichada", "Orinoquía", "Puerto Carreño", 6.1890, -67.4859, "CO-VID"),
]
CODIGOS_DEPARTAMENTO = [d[0] for d in DEPARTAMENTOS]

# Variantes de nombre (ya normalizadas con texto_normalizado) → nombre canónico.
# Solo se usa para verificar consistencia código↔nombre.
VARIANTES_DEPARTAMENTO = {
    "BOGOTA": "BOGOTA D C", "BOGOTA DC": "BOGOTA D C", "BOGOTA D C": "BOGOTA D C",
    "DISTRITO CAPITAL": "BOGOTA D C", "BOGOTA DISTRITO CAPITAL": "BOGOTA D C",
    "SAN ANDRES Y PROVIDENCIA": "SAN ANDRES", "ARCHIPIELAGO DE SAN ANDRES": "SAN ANDRES",
    "ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA": "SAN ANDRES",
    "SAN ANDRES PROVIDENCIA Y SANTA CATALINA": "SAN ANDRES",
    "VALLE": "VALLE DEL CAUCA", "GUAJIRA": "LA GUAJIRA", "NORTE SANTANDER": "NORTE DE SANTANDER",
}


def df_departamentos() -> DataFrame:
    esquema = "cod_dpto INT, nombre_canonico STRING, departamento STRING, region STRING, capital STRING, latitud DOUBLE, longitud DOUBLE, codigo_iso STRING"
    return (spark.createDataFrame(DEPARTAMENTOS, esquema)
            .withColumn("nombre_mapa", F.concat(F.col("departamento"), F.lit(", Colombia"))))


def homologar_nombre_departamento(col):
    c = texto_normalizado(col)
    expr = c
    for origen, destino in VARIANTES_DEPARTAMENTO.items():
        expr = F.when(c == origen, F.lit(destino)).otherwise(expr)
    return expr

## Contratos de datos por fuente
Cada columna canónica acepta una lista de alias (nombres técnicos tal como quedan tras normalizar el encabezado original).
Si una columna **obligatoria** no aparece, el pipeline se detiene con un mensaje explícito. Esto corrige la inconsistencia
`departamento` vs `departamento_ocurrencia` de SIVIGILA señalada en el informe.

In [ ]:
CONTRATOS = {
    "sivigila": {
        "patron_archivo": r"(?i)^sivigila.*\.csv$",
        "encoding": "UTF-8",
        "separador": ",",
        "decimal": ".",
        "columnas": {
            "cod_evento": ["cod_eve", "cod_evento", "codigo_evento"],
            "nombre_evento": ["nombre_evento", "nombre_del_evento", "evento"],
            "semana": ["semana", "semana_epidemiologica", "sem"],
            "anio": ["ano", "anio", "year", "a_o"],
            "cod_dpto": ["cod_dpto_o", "cod_departamento", "cod_dpto", "codigo_departamento"],
            "cod_mun": ["cod_mun_o", "cod_municipio", "cod_mun", "codigo_municipio"],
            "departamento_nombre": ["departamento_ocurrencia", "departamento", "nombre_departamento"],
            "municipio_nombre": ["municipio_ocurrencia", "municipio", "nombre_municipio"],
        },
        "obligatorias": ["nombre_evento", "anio", "cod_dpto"],
    },
    "pobreza": {
        "patron_archivo": r"(?i)^pobreza.*\.csv$",
        "encoding": "ISO-8859-1",
        "separador": ",",
        "decimal": ",",
        "columnas": {
            "cod_dpto": ["codigo_departamento", "c_digo_departamento", "cod_departamento"],
            "departamento_nombre": ["departamento"],
            "cod_entidad": ["codigo_entidad", "c_digo_entidad", "cod_entidad"],
            "entidad": ["entidad"],
            "dimension": ["dimension", "dimensi_n"],
            "subcategoria": ["subcategoria", "subcategor_a"],
            "indicador": ["indicador"],
            "dato_numerico": ["dato_numerico", "dato_num_rico"],
            "dato_cualitativo": ["dato_cualitativo"],
            "anio": ["ano", "anio", "a_o", "year"],
            "mes": ["mes"],
            "fuente": ["fuente"],
            "unidad_medida": ["unidad_de_medida", "unidad_medida"],
        },
        "obligatorias": ["cod_dpto", "cod_entidad", "indicador", "dato_numerico", "anio"],
    },
    "educacion": {
        "patron_archivo": r"(?i)^educacion.*\.csv$",
        "encoding": "UTF-8",
        "separador": ",",
        "decimal": "auto",
        "columnas": {
            "anio": ["ano", "anio", "a_o", "year"],
            "cod_dpto": ["codigo_departamento", "c_digo_departamento", "cod_departamento"],
            "departamento_nombre": ["departamento"],
            "poblacion_5_16": ["poblacion_5_16", "poblaci_n_5_16"],
            "tasa_matriculacion_5_16": ["tasa_matriculacion_5_16", "tasa_matriculaci_n_5_16"],
            "cobertura_neta": ["cobertura_neta"],
            "cobertura_neta_transicion": ["cobertura_neta_transicion", "cobertura_neta_transici_n"],
            "cobertura_neta_primaria": ["cobertura_neta_primaria"],
            "cobertura_neta_secundaria": ["cobertura_neta_secundaria"],
            "cobertura_neta_media": ["cobertura_neta_media"],
            "cobertura_bruta": ["cobertura_bruta"],
            "cobertura_bruta_transicion": ["cobertura_bruta_transicion", "cobertura_bruta_transici_n"],
            "cobertura_bruta_primaria": ["cobertura_bruta_primaria"],
            "cobertura_bruta_secundaria": ["cobertura_bruta_secundaria"],
            "cobertura_bruta_media": ["cobertura_bruta_media"],
            "tamano_promedio_de_grupo": ["tamano_promedio_de_grupo", "tama_o_promedio_de_grupo"],
            "sedes_conectadas_a_internet": ["sedes_conectadas_a_internet"],
            "tasa_desercion": ["desercion", "deserci_n", "tasa_desercion"],
            "desercion_transicion": ["desercion_transicion", "deserci_n_transici_n"],
            "desercion_primaria": ["desercion_primaria", "deserci_n_primaria"],
            "desercion_secundaria": ["desercion_secundaria", "deserci_n_secundaria"],
            "desercion_media": ["desercion_media", "deserci_n_media"],
            "aprobacion": ["aprobacion", "aprobaci_n"],
            "aprobacion_transicion": ["aprobacion_transicion", "aprobaci_n_transici_n"],
            "aprobacion_primaria": ["aprobacion_primaria", "aprobaci_n_primaria"],
            "aprobacion_secundaria": ["aprobacion_secundaria", "aprobaci_n_secundaria"],
            "aprobacion_media": ["aprobacion_media", "aprobaci_n_media"],
            "reprobacion": ["reprobacion", "reprobaci_n"],
            "reprobacion_transicion": ["reprobacion_transicion", "reprobaci_n_transici_n"],
            "reprobacion_primaria": ["reprobacion_primaria", "reprobaci_n_primaria"],
            "reprobacion_secundaria": ["reprobacion_secundaria", "reprobaci_n_secundaria"],
            "reprobacion_media": ["reprobacion_media", "reprobaci_n_media"],
            "repitencia": ["repitencia"],
            "repitencia_transicion": ["repitencia_transicion", "repitencia_transici_n"],
            "repitencia_primaria": ["repitencia_primaria"],
            "repitencia_secundaria": ["repitencia_secundaria"],
            "repitencia_media": ["repitencia_media"],
        },
        "obligatorias": ["anio", "cod_dpto", "tasa_desercion"],
    },
}


def aplicar_contrato(df: DataFrame, fuente: str, capa: str = "plata") -> DataFrame:
    """Renombra alias → nombre canónico, verifica obligatorias y devuelve solo columnas del contrato + técnicas."""
    contrato = CONTRATOS[fuente]
    disponibles = set(df.columns)
    seleccion, faltantes, mapeo = [], [], {}
    for canonica, alias in contrato["columnas"].items():
        encontrada = next((a for a in alias if a in disponibles), None)
        if encontrada is None:
            if canonica in contrato["obligatorias"]:
                faltantes.append(canonica)
            seleccion.append(F.lit(None).cast("string").alias(canonica))
        else:
            mapeo[canonica] = encontrada
            seleccion.append(F.col(encontrada).alias(canonica))
    verificar(capa, fuente, "contrato: columnas obligatorias presentes", len(faltantes) == 0,
              valor=len(faltantes), umbral=0, severidad="BLOQUEANTE", dimension="conformidad",
              detalle=f"faltantes={faltantes}; columnas disponibles={sorted(disponibles)}")
    opcionales_faltantes = [c for c in contrato["columnas"] if c not in mapeo]
    verificar(capa, fuente, "contrato: columnas opcionales presentes", len(opcionales_faltantes) == 0,
              valor=len(opcionales_faltantes), umbral=0, dimension="conformidad",
              detalle=f"ausentes (quedan nulas)={opcionales_faltantes}")
    tecnicas = [c for c in df.columns if c.startswith("_")]
    return df.select(*seleccion, *[F.col(c) for c in tecnicas])

## Clasificación epidemiológica de eventos SIVIGILA (objetivo específico 4)
Reglas ordenadas (la primera coincidencia gana) sobre el nombre normalizado del evento. El catálogo resultante se
persiste (`capa_oro.dim_evento_sivigila`) con el número de casos por evento para que el equipo audite la clasificación.

In [ ]:
REGLAS_PERFIL_EVENTO = [
    # (perfil, grupo, regex sobre nombre normalizado)
    ("NUTRICIONAL", "NUTRICIONAL", r"DESNUTRICION|BAJO PESO AL NACER|MALNUTRICION|NUTRICION"),
    ("ITS", "INFECCIOSA", r"SIFILIS|VIH|SIDA|HEPATITIS B|HEPATITIS C|HEPATITIS B C|TRANSMISION SEXUAL"),
    ("VECTORIAL", "INFECCIOSA", r"DENGUE|MALARIA|CHIKUNG|ZIKA|LEISHMANI|CHAGAS|FIEBRE AMARILLA|ENCEFALITIS"),
    ("RESPIRATORIA", "INFECCIOSA", r"RESPIRATORI|\bIRAG?\b|TUBERCULOSIS|COVID|TOS FERINA|INFLUENZA|NEUMONIA|SARS"),
    ("HIDRICA_ALIMENTARIA", "INFECCIOSA", r"DIARREIC|\bEDA\b|TRANSMITIDA POR ALIMENTOS|\bETA\b|HEPATITIS A|TIFOIDEA|PARATIFOIDEA|COLERA|ROTAVIRUS"),
    ("INMUNOPREVENIBLE", "INFECCIOSA", r"SARAMPION|RUBEOLA|PAROTIDITIS|VARICELA|DIFTERIA|TETANO|PARALISIS FLACIDA|MENINGITIS|EVENTO ADVERSO|ESAVI|VACUNA"),
    ("ZOONOSIS_OTRAS_INFECCIOSAS", "INFECCIOSA", r"RABIA|LEPTOSPIR|AGRESIONES POR ANIMALES|BRUCEL|LEPRA|HANSEN|ESTREPTOCOC|INFECCION|PAROTID|MICOSIS"),
    ("VIOLENCIA", "PSICOSOCIAL", r"VIOLENCIA|ABUSO|AGRESION SEXUAL|ATAQUE CON AGENTES|MALTRATO"),
    ("SALUD_MENTAL", "PSICOSOCIAL", r"SUICIDIO|CONSUMO DE SUSTANCIA|SUSTANCIAS PSICOACTIVAS|SALUD MENTAL"),
    ("MATERNO_PERINATAL", "MATERNO_PERINATAL", r"MATERN|PERINATAL|NEONATAL|GESTANT|EMBARAZ|CONGENIT"),
    ("LESIONES_INTOXICACIONES", "OTROS", r"INTOXICACION|LESION|POLVORA|OFIDICO|ACCIDENTE|QUEMADURA"),
    ("CRONICAS_CANCER", "OTROS", r"CANCER|LEUCEMIA|HUERFANA|TUMOR|CRONIC"),
]
PERFILES = [r[0] for r in REGLAS_PERFIL_EVENTO] + ["OTROS"]
GRUPOS_PERFIL = ["INFECCIOSA", "NUTRICIONAL", "PSICOSOCIAL", "MATERNO_PERINATAL", "OTROS"]


def clasificar_evento(col_nombre_normalizado):
    perfil = F.lit("OTROS")
    grupo = F.lit("OTROS")
    for p, g, rx in reversed(REGLAS_PERFIL_EVENTO):
        cond = col_nombre_normalizado.rlike(rx)
        perfil = F.when(cond, F.lit(p)).otherwise(perfil)
        grupo = F.when(cond, F.lit(g)).otherwise(grupo)
    return perfil, grupo

## Utilidades de panel temporal
Los rezagos se calculan por **unión con `anio - k`** y no con `F.lag`: si falta un año intermedio, `lag` tomaría
silenciosamente un año equivocado; la unión deja el rezago nulo, que es lo correcto.

In [ ]:
def agregar_rezagos(df: DataFrame, columnas, k: int = 1, llave="cod_dpto", sufijo=None) -> DataFrame:
    sufijo = sufijo or f"_lag{k}"
    prev = df.select(F.col(llave), (F.col("anio") + k).alias("anio"),
                     *[F.col(c).alias(f"{c}{sufijo}") for c in columnas])
    return df.join(prev, on=[llave, "anio"], how="left")


def verificar_unicidad(df: DataFrame, llaves, capa, tabla, severidad="BLOQUEANTE") -> int:
    dup = df.groupBy(*llaves).count().filter("count > 1").count()
    verificar(capa, tabla, f"unicidad de la llave {tuple(llaves)}", dup == 0, valor=dup, umbral=0,
              severidad=severidad, dimension="unicidad")
    return dup


print("Utilidades cargadas ✔")

Utilidades cargadas ✔


## 01_Capa_Bronce

<!-- merged from: 01_Capa_Bronce.ipynb (Jupy Tools) -->

# 01 · Capa Bronce — ingesta *raw* inmutable, idempotente y auditable

**CRISP-DM:** Comprensión de los datos (perfil inicial) · **Medallion:** Bronce

### Principios aplicados (y hallazgo del informe que corrige cada uno)
| Principio | Implementación | Hallazgo corregido |
|---|---|---|
| **El contenido no se modifica** | Todas las columnas se leen como `STRING` (sin `inferSchema`). Los valores quedan exactamente como vienen en el CSV. | `inferSchema` no reproducible; valores convertidos a nulos en silencio (los 730 nulos de SIVIGILA) |
| Nombres de columna técnicos con **linaje** | Delta exige nombres sin espacios/tildes: se normaliza el encabezado y se persiste la tabla `gobierno_datos.linaje_columnas` (nombre original ↔ nombre técnico ↔ posición). | "Bronze modifica nombres de columnas" sin trazabilidad |
| Registros corruptos visibles | Modo `PERMISSIVE` con `columnNameOfCorruptRecord = _corrupt_record` + regla de calidad con umbral. | `PERMISSIVE` ocultaba errores |
| **Idempotencia** | Tabla de control `gobierno_datos.control_cargas`: un archivo con la misma firma (nombre, tamaño, fecha de modificación) **no se vuelve a cargar**. Los nuevos lotes se **anexan** (`append`) con `_batch_id`. | `overwrite` destruía el histórico |
| Auditoría por registro | `_batch_id`, `_fecha_ingesta`, `_archivo_origen`, `_archivo_modificacion`, `_hash_registro` (SHA-256 del registro crudo). | Trazabilidad incompleta |
| Encoding por contrato | SIVIGILA y MEN en UTF-8; Terridata en ISO-8859-1 (definido en `CONTRATOS`). | Encabezados corruptos `c_digo_departamento` |
| Perfil persistido | `gobierno_datos.perfil_bronce`: % de nulos y cardinalidad aproximada por columna y lote. | "La exploración no se persiste" |

**Parámetro `modo_recarga`**: `incremental` (por defecto, idempotente) o `completa` (reconstruye Bronce desde cero; usar solo si cambia el contrato).

In [ ]:
%run ./00_Config_Utilidades

Entorno Databricks: True | formato: delta | ejecución: 20261005T153520-a98fe8


Utilidades cargadas ✔


In [ ]:
MODO_RECARGA = _widget("modo_recarga", "incremental").lower()
assert MODO_RECARGA in ("incremental", "completa"), "modo_recarga debe ser 'incremental' o 'completa'"
crear_esquemas()
print(f"Modo de recarga: {MODO_RECARGA} | origen: {INPUT_PATH}")

Modo de recarga: incremental | origen: /Volumes/desercion_escolar/capa_bronce/archivos_origen


## 1. Inventario de archivos de origen y asignación a su contrato

In [ ]:
def listar_archivos(ruta: str):
    """Devuelve [(nombre, ruta, tamaño, modificacion_ms)]. dbutils en Databricks; os fuera de él."""
    try:
        return [(f.name, f.path, int(f.size), int(getattr(f, "modificationTime", 0) or 0))
                for f in dbutils.fs.ls(ruta) if not f.name.endswith("/")]
    except (NameError, AttributeError):
        salida = []
        for nombre in sorted(os.listdir(ruta)):
            p = os.path.join(ruta, nombre)
            if os.path.isfile(p):
                st = os.stat(p)
                salida.append((nombre, p, int(st.st_size), int(st.st_mtime * 1000)))
        return salida


archivos = listar_archivos(INPUT_PATH)
inventario = []
for nombre, ruta, tamano, mod in archivos:
    fuente = next((f for f, c in CONTRATOS.items() if re.match(c["patron_archivo"], nombre)), None)
    inventario.append((fuente, nombre, ruta, tamano, mod))
    print(f"{'✔' if fuente else '✖ (sin contrato, se ignora)'} {nombre:35s} {tamano/1e6:9.2f} MB → {fuente}")

for fuente in CONTRATOS:
    n = sum(1 for i in inventario if i[0] == fuente)
    verificar("bronce", fuente, "existe al menos un archivo de origen", n > 0, valor=n, umbral=">=1",
              severidad="BLOQUEANTE", dimension="completitud")

✔ Educacion.csv                            0.14 MB → educacion
✔ Pobreza.csv                             18.71 MB → pobreza
✔ sivigila_2018-2024.csv                 199.51 MB → sivigila
✅ [bronce/sivigila] existe al menos un archivo de origen: valor=1 umbral=>=1 
✅ [bronce/pobreza] existe al menos un archivo de origen: valor=1 umbral=>=1 
✅ [bronce/educacion] existe al menos un archivo de origen: valor=1 umbral=>=1 


## 2. Tabla de control de cargas (idempotencia)

In [ ]:
ESQUEMA_CONTROL = T.StructType([
    T.StructField("fuente", T.StringType()), T.StructField("archivo", T.StringType()),
    T.StructField("ruta", T.StringType()), T.StructField("tamano_bytes", T.LongType()),
    T.StructField("modificacion_ms", T.LongType()), T.StructField("batch_id", T.StringType()),
    T.StructField("ejecucion_id", T.StringType()), T.StructField("fecha_carga", T.TimestampType()),
    T.StructField("encoding", T.StringType()), T.StructField("filas_cargadas", T.LongType()),
    T.StructField("filas_corruptas", T.LongType()), T.StructField("columnas_origen", T.IntegerType()),
    T.StructField("estado", T.StringType()),
])

if MODO_RECARGA == "completa" or not existe_tabla(SCHEMA_GOB, "control_cargas"):
    firmas_cargadas = set()
else:
    firmas_cargadas = {
        (r.fuente, r.archivo, r.tamano_bytes, r.modificacion_ms)
        for r in leer_tabla(SCHEMA_GOB, "control_cargas").filter("estado = 'OK'").collect()
    }
print(f"Firmas ya cargadas: {len(firmas_cargadas)}")

Firmas ya cargadas: 3


## 3. Función de ingesta *raw*

In [ ]:
def nombres_tecnicos(columnas_originales):
    vistos, salida = {}, []
    for c in columnas_originales:
        base = normalizar_nombre(c)
        n = vistos.get(base, 0)
        vistos[base] = n + 1
        salida.append(base if n == 0 else f"{base}_{n + 1}")   # colisiones explícitas, nunca silenciosas
    return salida


def ingerir_archivo(fuente: str, archivo: str, ruta: str, tamano: int, mod: int, primera_escritura: bool):
    contrato = CONTRATOS[fuente]
    batch_id = f"{fuente}-{EJECUCION_ID}"
    lector = (spark.read.option("header", True).option("encoding", contrato["encoding"])
              .option("sep", contrato["separador"]).option("quote", '"').option("escape", '"'))

    # 3.1 Encabezado original (sin inferir tipos: todo es STRING)
    originales = lector.csv(ruta).columns
    tecnicos = nombres_tecnicos(originales)
    esquema = T.StructType([T.StructField(c, T.StringType(), True) for c in tecnicos]
                           + [T.StructField("_corrupt_record", T.StringType(), True)])

    # 3.2 Lectura con esquema explícito y captura de registros corruptos
    df = (lector.schema(esquema).option("mode", "PERMISSIVE")
          .option("columnNameOfCorruptRecord", "_corrupt_record").csv(ruta))

    # 3.3 Columnas técnicas de auditoría (el contenido de negocio no se toca)
    df = (df
          .withColumn("_fuente", F.lit(fuente))
          .withColumn("_batch_id", F.lit(batch_id))
          .withColumn("_ejecucion_id", F.lit(EJECUCION_ID))
          .withColumn("_fecha_ingesta", F.current_timestamp())
          .withColumn("_archivo_origen", F.col("_metadata.file_path"))
          .withColumn("_archivo_modificacion", F.col("_metadata.file_modification_time"))
          .withColumn("_hash_registro", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("∅")) for c in tecnicos]), 256)))

    modo = "overwrite" if primera_escritura else "append"
    guardar_tabla(df, SCHEMA_BRONZE, fuente, modo=modo,
                  comentario=f"Bronce raw de {fuente}: valores originales como STRING + columnas de auditoria")

    # 3.4 Métricas del lote (leídas de la tabla persistida, no del CSV)
    lote = leer_tabla(SCHEMA_BRONZE, fuente).filter(F.col("_batch_id") == batch_id)
    filas = lote.count()
    corruptas = lote.filter(F.col("_corrupt_record").isNotNull()).count()
    vacias = lote.filter(F.coalesce(*[F.col(c) for c in tecnicos]).isNull()).count()
    duplicados_exactos = filas - lote.select(*tecnicos).distinct().count()

    verificar("bronce", fuente, "filas cargadas > 0", filas > 0, valor=filas, umbral=">0",
              severidad="BLOQUEANTE", dimension="completitud")
    verificar("bronce", fuente, "% registros corruptos", corruptas / max(filas, 1) < 0.01,
              valor=round(100 * corruptas / max(filas, 1), 4), umbral="<1%", dimension="validez",
              detalle=f"{corruptas} registros con _corrupt_record")
    verificar("bronce", fuente, "registros totalmente vacíos", vacias == 0, valor=vacias, umbral=0,
              dimension="completitud", detalle="se conservan en Bronce y se envían a cuarentena en Plata")
    # Informativo: en SIVIGILA cada fila es una notificación sin identificador de caso,
    # por lo que filas idénticas son casos distintos legítimos (NO se eliminan en Plata).
    verificar("bronce", fuente, "filas idénticas (informativo)", True, valor=duplicados_exactos,
              dimension="unicidad", detalle="no se interpretan como duplicados sin una llave de negocio")

    # 3.5 Linaje de columnas
    linaje = spark.createDataFrame(
        [(fuente, batch_id, i + 1, o, t) for i, (o, t) in enumerate(zip(originales, tecnicos))],
        "fuente STRING, batch_id STRING, posicion INT, nombre_original STRING, nombre_tecnico STRING")
    guardar_tabla(linaje, SCHEMA_GOB, "linaje_columnas", modo="append")

    # 3.6 Perfil por columna (persistido)
    total = max(filas, 1)
    agg = lote.select(
        *[F.sum(F.col(c).isNull().cast("int")).alias(f"n__{c}") for c in tecnicos],
        *[F.approx_count_distinct(F.col(c)).alias(f"d__{c}") for c in tecnicos]).first()
    perfil = spark.createDataFrame(
        [(fuente, batch_id, c, float(agg[f"n__{c}"] or 0) / total, int(agg[f"d__{c}"] or 0)) for c in tecnicos],
        "fuente STRING, batch_id STRING, columna STRING, pct_nulos DOUBLE, cardinalidad_aprox BIGINT")
    guardar_tabla(perfil, SCHEMA_GOB, "perfil_bronce", modo="append")
    display(perfil)

    return (fuente, archivo, ruta, tamano, mod, batch_id, EJECUCION_ID, datetime.now(),
            contrato["encoding"], filas, corruptas, len(originales), "OK")

## 4. Ejecución de la ingesta

In [ ]:
registros_control = []
for fuente, archivo, ruta, tamano, mod in inventario:
    if fuente is None:
        continue
    if (fuente, archivo, tamano, mod) in firmas_cargadas:
        print(f"↷ {archivo}: ya cargado con la misma firma → se omite (idempotencia)")
        continue
    primera = MODO_RECARGA == "completa" and not any(r[0] == fuente for r in registros_control)
    primera = primera or not existe_tabla(SCHEMA_BRONZE, fuente)
    print(f"\n▶ Ingestando {archivo} ({fuente}) modo={'overwrite' if primera else 'append'}")
    registros_control.append(ingerir_archivo(fuente, archivo, ruta, tamano, mod, primera))

if registros_control:
    modo_control = "overwrite" if (MODO_RECARGA == "completa" or not existe_tabla(SCHEMA_GOB, "control_cargas")) else "append"
    guardar_tabla(spark.createDataFrame(registros_control, ESQUEMA_CONTROL), SCHEMA_GOB, "control_cargas", modo=modo_control)
else:
    print("No hay archivos nuevos: Bronce ya está al día.")

persistir_calidad()

↷ Educacion.csv: ya cargado con la misma firma → se omite (idempotencia)
↷ Pobreza.csv: ya cargado con la misma firma → se omite (idempotencia)
↷ sivigila_2018-2024.csv: ya cargado con la misma firma → se omite (idempotencia)
No hay archivos nuevos: Bronce ya está al día.
  ✔ desercion_escolar.gobierno_datos.control_calidad (append) → 131 filas


## 5. Validación de la capa Bronce

In [ ]:
display(leer_tabla(SCHEMA_GOB, "control_cargas").orderBy(F.desc("fecha_carga")))
for fuente in CONTRATOS:
    t = leer_tabla(SCHEMA_BRONZE, fuente)
    print(f"{fqtn(SCHEMA_BRONZE, fuente):45s} total={t.count():>10,}  lotes={t.select('_batch_id').distinct().count()}")
display(leer_tabla(SCHEMA_GOB, "linaje_columnas").orderBy("fuente", "posicion"))

fuente,archivo,ruta,tamano_bytes,modificacion_ms,batch_id,ejecucion_id,fecha_carga,encoding,filas_cargadas,filas_corruptas,columnas_origen,estado
sivigila,sivigila_2018-2024.csv,dbfs:/Volumes/desercion_escolar/capa_bronce/archivos_origen/sivigila_2018-2024.csv,199507906,1791164725000,sivigila-20261005T014216-7ce4f8,20261005T014216-7ce4f8,2026-10-05T01:54:40.288Z,UTF-8,3056235,0,8,OK
pobreza,Pobreza.csv,dbfs:/Volumes/desercion_escolar/capa_bronce/archivos_origen/Pobreza.csv,18713896,1791164716000,pobreza-20261005T014216-7ce4f8,20261005T014216-7ce4f8,2026-10-05T01:54:18.796Z,ISO-8859-1,83148,0,13,OK
educacion,Educacion.csv,dbfs:/Volumes/desercion_escolar/capa_bronce/archivos_origen/Educacion.csv,141013,1791164715000,educacion-20261005T014216-7ce4f8,20261005T014216-7ce4f8,2026-10-05T01:54:02.421Z,UTF-8,462,0,37,OK


desercion_escolar.capa_bronce.sivigila        total= 3,056,235  lotes=1
desercion_escolar.capa_bronce.pobreza         total=    83,148  lotes=1
desercion_escolar.capa_bronce.educacion       total=       462  lotes=1


fuente,batch_id,posicion,nombre_original,nombre_tecnico
educacion,educacion-20261005T014216-7ce4f8,1,AÑO,ano
educacion,educacion-20261005T014216-7ce4f8,2,CÓDIGO_DEPARTAMENTO,codigo_departamento
educacion,educacion-20261005T014216-7ce4f8,3,DEPARTAMENTO,departamento
educacion,educacion-20261005T014216-7ce4f8,4,POBLACIÓN_5_16,poblacion_5_16
educacion,educacion-20261005T014216-7ce4f8,5,TASA_MATRICULACIÓN_5_16,tasa_matriculacion_5_16
educacion,educacion-20261005T014216-7ce4f8,6,COBERTURA_NETA,cobertura_neta
educacion,educacion-20261005T014216-7ce4f8,7,COBERTURA_NETA_TRANSICIÓN,cobertura_neta_transicion
educacion,educacion-20261005T014216-7ce4f8,8,COBERTURA_NETA_PRIMARIA,cobertura_neta_primaria
educacion,educacion-20261005T014216-7ce4f8,9,COBERTURA_NETA_SECUNDARIA,cobertura_neta_secundaria
educacion,educacion-20261005T014216-7ce4f8,10,COBERTURA_NETA_MEDIA,cobertura_neta_media


## 02_Capa_Plata

<!-- merged from: 02_Capa_Plata.ipynb (Jupy Tools) -->

# 02 · Capa Plata — limpieza, contratos, homologación DIVIPOLA y calidad bloqueante

**CRISP-DM:** Preparación de los datos · **Medallion:** Plata

| Fuente | Decisión técnica | Hallazgo corregido |
|---|---|---|
| Todas | Solo se procesa el **lote vigente** de Bronce (último lote OK en `control_cargas`). | Reprocesos sobre datos mezclados |
| Todas | **Contrato de columnas** (alias → nombre canónico) con columnas obligatorias bloqueantes. | `departamento` vs `departamento_ocurrencia` |
| Todas | Llave territorial = **código DIVIPOLA**; el nombre solo se usa para verificar consistencia. Registros fuera de los 33 departamentos van a **cuarentena** con motivo. | Homologación por texto; `EXTERIOR`/`PROCEDENCIA DESCONOCIDA` sin tratamiento |
| Todas | *Parsing* numérico ANSI-safe por contrato de formato + conteo de valores no parseables. | Conversión numérica riesgosa |
| SIVIGILA | **No se eliminan filas idénticas**: cada fila es una notificación sin identificador de caso; dos casos del mismo evento, municipio y semana son idénticos y legítimos. | `dropDuplicates()` eliminaba **≈49 %** de los casos (3.055.505 → 1.564.570) |
| SIVIGILA | Clasificación de eventos en perfiles (infecciosos, nutricionales, psicosociales, materno-perinatales). | Objetivo específico 4 no implementado |
| SIVIGILA | Ausencia de reporte ≠ cero: panel completo con bandera `sivigila_reporto`. | Nulos imputados a 0 sin distinguir |
| Terridata | Se conservan **solo filas de nivel departamental** (`cod_entidad = cod_dpto × 1000`). | El pivot promediaba municipios con su departamento |
| Terridata | Tabla **larga** con `indicador_id` estable + catálogo; el pivot se hace en Oro con lista explícita. | Pivot dinámico, esquema cambiante, IPM duplicado por tildes |
| Terridata | Deduplicación por llave de negocio (departamento, indicador, año) quedándose con el último mes. | `avg` mezclando escalas/meses |
| MEN | Se conserva 2011–2024 para construir rezagos de 2018 sin imputar. | Primer año del panel con rezago nulo |
| MEN | Pruebas de coherencia (aprobación + reprobación + deserción ≈ 100) y detección de **columnas desalineadas** en la fuente. | Desagregados por nivel inconsistentes sin detectar |

In [ ]:
%run ./00_Config_Utilidades

Entorno Databricks: True | formato: delta | ejecución: 20261005T153646-921cd2


Utilidades cargadas ✔


In [ ]:
crear_esquemas()
dim_dpto = df_departamentos()


def lote_vigente(fuente: str):
    ctrl = (leer_tabla(SCHEMA_GOB, "control_cargas")
            .filter((F.col("fuente") == fuente) & (F.col("estado") == "OK"))
            .orderBy(F.desc("fecha_carga")))
    ultimo = ctrl.first()
    verificar("plata", fuente, "existe lote vigente en Bronce", ultimo is not None, severidad="BLOQUEANTE",
              dimension="completitud")
    df = leer_tabla(SCHEMA_BRONZE, fuente).filter(F.col("_batch_id") == ultimo.batch_id)
    print(f"{fuente}: lote vigente {ultimo.batch_id} ({ultimo.archivo}, {ultimo.filas_cargadas:,} filas)")
    return df, ultimo.batch_id

## 1. SIVIGILA

In [ ]:
raw_siv, batch_siv = lote_vigente("sivigila")
n_raw_siv = raw_siv.count()
siv = aplicar_contrato(raw_siv, "sivigila")

siv = siv.select(
    parsear_entero("cod_evento").alias("cod_evento"),
    F.trim(F.col("nombre_evento")).alias("nombre_evento"),
    texto_normalizado("nombre_evento").alias("evento_normalizado"),
    parsear_entero("semana").alias("semana_raw"),
    parsear_entero("anio").alias("anio"),
    parsear_entero("cod_dpto").alias("cod_dpto_raw"),
    parsear_entero("cod_mun").alias("cod_mun"),
    homologar_nombre_departamento("departamento_nombre").alias("departamento_reportado"),
    texto_normalizado("municipio_nombre").alias("municipio"),
    F.col("cod_evento").alias("_cod_evento_raw"), F.col("anio").alias("_anio_raw"),
    F.col("cod_dpto").alias("_cod_dpto_raw"), F.col("semana").alias("_semana_raw"),
    "_corrupt_record", "_batch_id", "_hash_registro",
)

fallas = contar_fallas_parseo(siv, [("_anio_raw", "anio"), ("_cod_dpto_raw", "cod_dpto_raw"),
                                    ("_semana_raw", "semana_raw"), ("_cod_evento_raw", "cod_evento")])
for col_p, n in fallas.items():
    verificar("plata", "sivigila", f"valores no parseables en {col_p}", n / n_raw_siv < 0.001,
              valor=n, umbral="<0.1%", dimension="validez")

# Recuperación del código a partir del nombre cuando el código falta (y viceversa: verificación cruzada)
siv = (siv.join(dim_dpto.select(F.col("cod_dpto").alias("_cod_por_nombre"),
                                F.col("nombre_canonico").alias("departamento_reportado")),
                on="departamento_reportado", how="left")
       .withColumn("cod_dpto", F.coalesce(F.col("cod_dpto_raw"), F.col("_cod_por_nombre")))
       .withColumn("semana", F.when(F.col("semana_raw").between(1, 53), F.col("semana_raw")))
       .join(dim_dpto.select("cod_dpto", F.col("nombre_canonico").alias("_nombre_dim")), on="cod_dpto", how="left"))

perfil, grupo = clasificar_evento(F.col("evento_normalizado"))
siv = siv.withColumn("perfil_evento", perfil).withColumn("grupo_perfil", grupo)

motivo = (F.when(F.col("_corrupt_record").isNotNull(), "registro_corrupto")
          .when(F.col("anio").isNull() & F.col("cod_dpto").isNull() & F.col("nombre_evento").isNull(), "registro_vacio")
          .when(F.col("anio").isNull(), "anio_nulo")
          .when(~F.col("anio").between(ANIO_MIN, ANIO_MAX), "anio_fuera_de_periodo")
          .when(F.col("_nombre_dim").isNull(), "departamento_no_valido")
          .when(F.col("nombre_evento").isNull(), "evento_nulo"))
siv = cachear(siv.withColumn("motivo_cuarentena", motivo))

resumen_motivos = {r["motivo_cuarentena"]: r["count"] for r in siv.groupBy("motivo_cuarentena").count().collect()}
print("Distribución de motivos de cuarentena:", resumen_motivos)

n_validos = resumen_motivos.get(None, 0)
n_cuarentena = n_raw_siv - n_validos - resumen_motivos.get("anio_fuera_de_periodo", 0)
verificar("plata", "sivigila", "% registros en cuarentena", n_cuarentena / n_raw_siv < 0.02,
          valor=round(100 * n_cuarentena / n_raw_siv, 3), umbral="<2%", dimension="validez",
          detalle=str(resumen_motivos))
verificar("plata", "sivigila", "conservación de registros (válidos + cuarentena + fuera de periodo = bronce)",
          sum(resumen_motivos.values()) == n_raw_siv, valor=sum(resumen_motivos.values()), umbral=n_raw_siv,
          severidad="BLOQUEANTE", dimension="integridad")

validos = siv.filter(F.col("motivo_cuarentena").isNull())
inconsistentes = validos.filter(F.col("departamento_reportado").isNotNull()
                                & (F.col("departamento_reportado") != F.col("_nombre_dim"))).count()
verificar("plata", "sivigila", "consistencia código DIVIPOLA ↔ nombre de departamento",
          inconsistentes / max(n_validos, 1) < 0.01, valor=inconsistentes, umbral="<1%",
          dimension="consistencia")
mun_inconsistentes = validos.filter(F.col("cod_mun").isNotNull()
                                    & ((F.col("cod_mun") / 1000).cast("int") != F.col("cod_dpto"))).count()
verificar("plata", "sivigila", "consistencia municipio ↔ departamento (cod_mun // 1000 = cod_dpto)",
          mun_inconsistentes / max(n_validos, 1) < 0.01, valor=mun_inconsistentes, umbral="<1%",
          dimension="consistencia")
verificar("plata", "sivigila", "semanas epidemiológicas inválidas (se anulan, el caso se conserva)",
          True, valor=validos.filter(F.col("semana").isNull()).count(), dimension="validez")
pct_otros = validos.filter(F.col("perfil_evento") == "OTROS").count() / max(n_validos, 1)
verificar("plata", "sivigila", "% casos sin perfil epidemiológico (OTROS)", pct_otros < 0.25,
          valor=round(100 * pct_otros, 2), umbral="<25%", dimension="validez",
          detalle="revisar REGLAS_PERFIL_EVENTO si es alto")

✅ [plata/sivigila] existe lote vigente en Bronce: valor=None umbral=None 
sivigila: lote vigente sivigila-20261005T014216-7ce4f8 (sivigila_2018-2024.csv, 3,056,235 filas)
✅ [plata/sivigila] contrato: columnas obligatorias presentes: valor=0 umbral=0 faltantes=[]; columnas disponibles=['_archivo_modificacion', '_archivo_origen', '_batch_id', '_corrupt_record', '_ejecucion_id', '_fecha_ingesta', '_fuente', '_hash_registro', 'ano', 'cod_departamento', 'cod_evento', 'cod_municipio', 'departamento_ocurrencia', 'municipio_ocurrencia', 'nombre_evento', 'semana']
✅ [plata/sivigila] contrato: columnas opcionales presentes: valor=0 umbral=0 ausentes (quedan nulas)=[]
✅ [plata/sivigila] valores no parseables en anio: valor=0 umbral=<0.1% 
✅ [plata/sivigila] valores no parseables en cod_dpto_raw: valor=0 umbral=<0.1% 
✅ [plata/sivigila] valores no parseables en semana_raw: valor=0 umbral=<0.1% 
✅ [plata/sivigila] valores no parseables en cod_evento: valor=0 umbral=<0.1% 
Distribución de motivos de

True

In [ ]:
sivigila_limpio = validos.select(
    "cod_dpto", F.col("_nombre_dim").alias("departamento"), "cod_mun", "municipio", "anio", "semana",
    "cod_evento", "nombre_evento", "perfil_evento", "grupo_perfil", "_batch_id", "_hash_registro")
guardar_tabla(sivigila_limpio, SCHEMA_SILVER, "sivigila_limpio", particion="anio",
              comentario="SIVIGILA a nivel de notificacion; sin deduplicar (cada fila es un caso)")

guardar_tabla(siv.filter(F.col("motivo_cuarentena").isNotNull() & (F.col("motivo_cuarentena") != "anio_fuera_de_periodo"))
              .select("motivo_cuarentena", "_cod_evento_raw", "nombre_evento", "_anio_raw", "_semana_raw",
                      "_cod_dpto_raw", "departamento_reportado", "municipio", "_corrupt_record", "_batch_id", "_hash_registro"),
              SCHEMA_SILVER, "sivigila_cuarentena")

# Catálogo de eventos (auditable)
catalogo_eventos = (sivigila_limpio.groupBy("cod_evento", "nombre_evento", "perfil_evento", "grupo_perfil")
                    .agg(F.count("*").alias("casos_2018_2024")))
guardar_tabla(catalogo_eventos, SCHEMA_SILVER, "sivigila_catalogo_eventos")
display(catalogo_eventos.orderBy(F.desc("casos_2018_2024")))

  ✔ desercion_escolar.capa_plata.sivigila_limpio (overwrite) → 3,047,498 filas
  ✔ desercion_escolar.capa_plata.sivigila_cuarentena (overwrite) → 8,737 filas
  ✔ desercion_escolar.capa_plata.sivigila_catalogo_eventos (overwrite) → 82 filas


cod_evento,nombre_evento,perfil_evento,grupo_perfil,casos_2018_2024
300,AGRESIONES POR ANIMALES POTENCIALMENTE TRANSMISORES DE RABIA,ZOONOSIS_OTRAS_INFECCIOSAS,INFECCIOSA,505380
210,DENGUE,VECTORIAL,INFECCIOSA,499217
875,VCM VIF VSX,OTROS,OTROS,447765
490,MALARIA VIVAX,VECTORIAL,INFECCIOSA,166642
356,INTENTO DE SUICIDIO,SALUD_MENTAL,PSICOSOCIAL,136584
549,MORBILIDAD MATERNA EXTREMA,MATERNO_PERINATAL,MATERNO_PERINATAL,115771
831,VARICELA INDIVIDUAL,INMUNOPREVENIBLE,INFECCIOSA,106585
470,MALARIA FALCIPARUM,VECTORIAL,INFECCIOSA,98800
348,INFECCION RESPIRATORIA AGUDA GRAVE IRAG INUSITADA,RESPIRATORIA,INFECCIOSA,92664
370,INTOXICACION POR FARMACOS,LESIONES_INTOXICACIONES,OTROS,79363


### Agregación departamento-año (panel completo, ausencia de reporte ≠ cero)

In [ ]:
panel = dim_dpto.select("cod_dpto").crossJoin(spark.range(ANIO_MIN, ANIO_MAX + 1).select(F.col("id").cast("int").alias("anio")))

siv_perfil = (sivigila_limpio.groupBy("cod_dpto", "anio", "perfil_evento", "grupo_perfil")
              .agg(F.count("*").alias("casos")))
guardar_tabla(siv_perfil, SCHEMA_SILVER, "sivigila_departamento_anio_perfil", particion="anio")

base = (sivigila_limpio.groupBy("cod_dpto", "anio").agg(
    F.count("*").alias("casos_total"),
    F.countDistinct("cod_evento", "nombre_evento").alias("tipos_evento"),
    F.countDistinct("semana").alias("semanas_con_reporte"),
    F.countDistinct("cod_mun").alias("municipios_con_reporte"),
    *[F.sum((F.col("grupo_perfil") == g).cast("int")).alias(f"casos_grupo_{g.lower()}") for g in GRUPOS_PERFIL],
    *[F.sum((F.col("perfil_evento") == p).cast("int")).alias(f"casos_perfil_{p.lower()}") for p in PERFILES],
))
siv_anual = (panel.join(base, on=["cod_dpto", "anio"], how="left")
             .withColumn("sivigila_reporto", F.col("casos_total").isNotNull()))

no_reporta = siv_anual.filter(~F.col("sivigila_reporto")).count()
verificar("plata", "sivigila_departamento_anio", "departamento-año sin ningún reporte SIVIGILA",
          no_reporta == 0, valor=no_reporta, umbral=0, dimension="completitud",
          detalle="se conservan como NULO (no cero) para no confundir falta de información con ausencia de eventos")
verificar_unicidad(siv_anual, ["cod_dpto", "anio"], "plata", "sivigila_departamento_anio")

semanas_anio = {r["anio"]: r["s"] for r in sivigila_limpio.groupBy("anio").agg(F.countDistinct("semana").alias("s")).collect()}
for a in range(ANIO_MIN, ANIO_MAX + 1):
    verificar("plata", "sivigila", f"cobertura de semanas epidemiológicas {a}", semanas_anio.get(a, 0) >= 50,
              valor=semanas_anio.get(a, 0), umbral=">=50", dimension="completitud",
              detalle="un año incompleto subestima la carga de morbilidad (frecuente en el último año publicado)")

guardar_tabla(siv_anual, SCHEMA_SILVER, "sivigila_departamento_anio", particion="anio",
              comentario="SIVIGILA agregado por departamento-anio y perfil epidemiologico")
liberar(siv)

  ✔ desercion_escolar.capa_plata.sivigila_departamento_anio_perfil (overwrite) → 2,673 filas
✅ [plata/sivigila_departamento_anio] departamento-año sin ningún reporte SIVIGILA: valor=0 umbral=0 se conservan como NULO (no cero) para no confundir falta de información con ausencia de eventos
✅ [plata/sivigila_departamento_anio] unicidad de la llave ('cod_dpto', 'anio'): valor=0 umbral=0 
✅ [plata/sivigila] cobertura de semanas epidemiológicas 2018: valor=52 umbral=>=50 un año incompleto subestima la carga de morbilidad (frecuente en el último año publicado)
✅ [plata/sivigila] cobertura de semanas epidemiológicas 2019: valor=52 umbral=>=50 un año incompleto subestima la carga de morbilidad (frecuente en el último año publicado)
✅ [plata/sivigila] cobertura de semanas epidemiológicas 2020: valor=53 umbral=>=50 un año incompleto subestima la carga de morbilidad (frecuente en el último año publicado)
✅ [plata/sivigila] cobertura de semanas epidemiológicas 2021: valor=52 umbral=>=50 un año inco

## 2. Terridata — dimensión pobreza

In [ ]:
raw_pob, batch_pob = lote_vigente("pobreza")
n_raw_pob = raw_pob.count()
pob = aplicar_contrato(raw_pob, "pobreza")
pob = pob.select(
    parsear_entero("cod_dpto").alias("cod_dpto"),
    parsear_entero("cod_entidad").alias("cod_entidad"),
    F.trim(F.col("entidad")).alias("entidad"),
    homologar_nombre_departamento("departamento_nombre").alias("departamento_reportado"),
    F.trim(F.col("dimension")).alias("dimension"),
    F.trim(F.col("subcategoria")).alias("subcategoria"),
    F.trim(F.col("indicador")).alias("indicador"),
    slug_udf(F.trim(F.col("indicador"))).alias("indicador_id"),
    F.col("dato_numerico").alias("valor_original"),
    parsear_numero("dato_numerico", decimal=CONTRATOS["pobreza"]["decimal"]).alias("valor"),
    parsear_entero("anio").alias("anio"),
    parsear_entero("mes").alias("mes"),
    F.trim(F.col("fuente")).alias("fuente"),
    F.trim(F.col("unidad_medida")).alias("unidad_medida"),
    "_corrupt_record", "_batch_id",
)

fallas = contar_fallas_parseo(pob, [("valor_original", "valor")])
verificar("plata", "pobreza", "valores numéricos no parseables", fallas["valor"] / n_raw_pob < 0.005,
          valor=fallas["valor"], umbral="<0.5%", dimension="validez")
verificar("plata", "pobreza", "dato numérico nulo en origen (informativo)", True,
          valor=pob.filter(F.col("valor_original").isNull()).count(), dimension="completitud")

# Nivel territorial: Colombia (cod_dpto = 1) se separa como referencia nacional
nacional = pob.filter(F.col("cod_dpto") == 1)
es_dpto = F.col("cod_entidad") == F.col("cod_dpto") * 1000
es_bogota = (F.col("cod_dpto") == 11) & (F.col("cod_entidad") == 11001)
pob_dpto = (pob.filter(F.col("cod_dpto").isin(CODIGOS_DEPARTAMENTO) & (es_dpto | es_bogota))
            .withColumn("_prioridad_entidad", F.when(es_dpto, 0).otherwise(1)))

n_municipales = pob.filter(F.col("cod_dpto").isin(CODIGOS_DEPARTAMENTO) & ~(es_dpto | es_bogota)).count()
verificar("plata", "pobreza", "filas municipales excluidas (no se mezclan con el departamento)", True,
          valor=n_municipales, dimension="consistencia",
          detalle="la versión anterior promediaba municipios y departamento en el pivot")

# Rango de porcentajes
es_pct = F.lower(F.coalesce(F.col("unidad_medida"), F.lit(""))).contains("porcentaje")
fuera = pob_dpto.filter(es_pct & F.col("valor").isNotNull() & ~F.col("valor").between(0, 100)).count()
verificar("plata", "pobreza", "porcentajes fuera de [0, 100] (se anulan)", fuera == 0, valor=fuera, umbral=0,
          dimension="validez")
pob_dpto = pob_dpto.withColumn("valor", F.when(es_pct & ~F.col("valor").between(0, 100), None).otherwise(F.col("valor")))

# Consistencia nombre ↔ código
inc = (pob_dpto.join(dim_dpto.select("cod_dpto", "nombre_canonico"), "cod_dpto")
       .filter(F.col("departamento_reportado").isNotNull() & (F.col("departamento_reportado") != F.col("nombre_canonico")))
       .select("departamento_reportado", "nombre_canonico").distinct())
verificar("plata", "pobreza", "consistencia código DIVIPOLA ↔ nombre", inc.count() == 0, valor=inc.count(), umbral=0,
          dimension="consistencia", detalle=str([tuple(r) for r in inc.limit(10).collect()]))

# Colisiones de slug (p. ej. "Índice ..." vs "Indice ...") → se reportan y se resuelven en la deduplicación
colisiones = pob_dpto.groupBy("indicador_id").agg(F.countDistinct("indicador").alias("n")).filter("n > 1")
verificar("plata", "pobreza", "indicadores con nombres variantes que colapsan en el mismo id", True,
          valor=colisiones.count(), dimension="unicidad",
          detalle="resuelto: un único indicador_id estable (corrige IPM duplicado por tildes)")

# Deduplicación por llave de negocio: departamento-indicador-año; prioridad entidad departamental, último mes, valor no nulo
w = Window.partitionBy("cod_dpto", "indicador_id", "anio").orderBy(
    F.col("_prioridad_entidad"), F.col("valor").isNull(), F.desc_nulls_last("mes"))
pob_dpto_anio = (pob_dpto.filter(F.col("anio").between(ANIO_MIN_POBREZA, ANIO_MAX))
                 .withColumn("_r", F.row_number().over(w)).filter("_r = 1")
                 .select("cod_dpto", "anio", "mes", "indicador_id", "indicador", "dimension", "subcategoria",
                         "valor", "unidad_medida", "fuente", "_batch_id"))
verificar_unicidad(pob_dpto_anio, ["cod_dpto", "indicador_id", "anio"], "plata", "pobreza_departamento_indicador")

guardar_tabla(pob_dpto_anio, SCHEMA_SILVER, "pobreza_departamento_indicador", particion="anio",
              comentario="Terridata pobreza, nivel departamental, formato largo (una fila por depto-indicador-anio)")

w_n = Window.partitionBy("indicador_id", "anio").orderBy(F.desc_nulls_last("mes"))
guardar_tabla(nacional.filter(F.col("anio").between(ANIO_MIN_POBREZA, ANIO_MAX))
              .withColumn("_r", F.row_number().over(w_n)).filter("_r = 1")
              .select("anio", "indicador_id", "indicador", "valor", "unidad_medida", "fuente"),
              SCHEMA_SILVER, "pobreza_nacional_indicador")

# Catálogo de indicadores con cobertura (insumo de la selección explícita de variables en Oro)
catalogo_pob = (pob_dpto_anio.filter(F.col("valor").isNotNull()).groupBy("indicador_id").agg(
    F.first("indicador").alias("indicador"), F.first("dimension").alias("dimension"),
    F.first("subcategoria").alias("subcategoria"), F.first("unidad_medida").alias("unidad_medida"),
    F.first("fuente").alias("fuente"), F.countDistinct("cod_dpto").alias("n_departamentos"),
    F.min("anio").alias("anio_min"), F.max("anio").alias("anio_max"),
    F.countDistinct("anio").alias("n_anios"),
    F.sort_array(F.collect_set("anio")).alias("anios"))
    .withColumn("pct_cobertura_departamentos", F.col("n_departamentos") / F.lit(len(CODIGOS_DEPARTAMENTO)))
    .withColumn("es_estructural", F.col("n_anios") == 1))
guardar_tabla(catalogo_pob, SCHEMA_SILVER, "pobreza_catalogo_indicadores")
display(catalogo_pob.orderBy(F.desc("n_departamentos"), "indicador_id"))

✅ [plata/pobreza] existe lote vigente en Bronce: valor=None umbral=None 
pobreza: lote vigente pobreza-20261005T014216-7ce4f8 (Pobreza.csv, 83,148 filas)
✅ [plata/pobreza] contrato: columnas obligatorias presentes: valor=0 umbral=0 faltantes=[]; columnas disponibles=['_archivo_modificacion', '_archivo_origen', '_batch_id', '_corrupt_record', '_ejecucion_id', '_fecha_ingesta', '_fuente', '_hash_registro', 'ano', 'codigo_departamento', 'codigo_entidad', 'dato_cualitativo', 'dato_numerico', 'departamento', 'dimension', 'entidad', 'fuente', 'indicador', 'mes', 'subcategoria', 'unidad_de_medida']
✅ [plata/pobreza] contrato: columnas opcionales presentes: valor=0 umbral=0 ausentes (quedan nulas)=[]
✅ [plata/pobreza] valores numéricos no parseables: valor=0 umbral=<0.5% 
✅ [plata/pobreza] dato numérico nulo en origen (informativo): valor=1025 umbral=None 
✅ [plata/pobreza] filas municipales excluidas (no se mezclan con el departamento): valor=60620 umbral=None la versión anterior promediaba m

indicador_id,indicador,dimension,subcategoria,unidad_medida,fuente,n_departamentos,anio_min,anio_max,n_anios,anios,pct_cobertura_departamentos,es_estructural
acceso_inadecuado_a_eliminacion_de_excretas,Acceso inadecuado a eliminacion de excretas,Pobreza,Privaciones por hogar del IPM,Porcentaje (el valor esta multiplicado por 100),DANE - Encuesta Nacional de Calidad de Vida (ECV),33,2018,2024,7,"List(2018, 2019, 2020, 2021, 2022, 2023, 2024)",1.0,false
acceso_inadecuado_a_eliminacion_de_excretas_en_cabecera,Acceso inadecuado a eliminacion de excretas en Cabecera,Pobreza,Privaciones por hogar del IPM,Porcentaje (el valor esta multiplicado por 100),DANE - Encuesta Nacional de Calidad de Vida (ECV),33,2018,2024,7,"List(2018, 2019, 2020, 2021, 2022, 2023, 2024)",1.0,false
analfabetismo,Analfabetismo,Pobreza,Privaciones por hogar del IPM,Porcentaje (el valor esta multiplicado por 100),DANE - Encuesta Nacional de Calidad de Vida (ECV),33,2018,2024,7,"List(2018, 2019, 2020, 2021, 2022, 2023, 2024)",1.0,false
analfabetismo_en_cabecera,Analfabetismo en Cabecera,Pobreza,Privaciones por hogar del IPM,Porcentaje (el valor esta multiplicado por 100),DANE - Encuesta Nacional de Calidad de Vida (ECV),33,2018,2024,7,"List(2018, 2019, 2020, 2021, 2022, 2023, 2024)",1.0,false
bajo_logro_educativo,Bajo logro educativo,Pobreza,Privaciones por hogar del IPM,Porcentaje (el valor esta multiplicado por 100),DANE - Encuesta Nacional de Calidad de Vida (ECV),33,2018,2024,7,"List(2018, 2019, 2020, 2021, 2022, 2023, 2024)",1.0,false
bajo_logro_educativo_en_cabecera,Bajo logro educativo en Cabecera,Pobreza,Privaciones por hogar del IPM,Porcentaje (el valor esta multiplicado por 100),DANE - Encuesta Nacional de Calidad de Vida (ECV),33,2018,2024,7,"List(2018, 2019, 2020, 2021, 2022, 2023, 2024)",1.0,false
barreras_de_acceso_a_servicios_de_salud,Barreras de acceso a servicios de salud,Pobreza,Privaciones por hogar del IPM,Porcentaje (el valor esta multiplicado por 100),DANE - Encuesta Nacional de Calidad de Vida (ECV),33,2018,2024,7,"List(2018, 2019, 2020, 2021, 2022, 2023, 2024)",1.0,false
barreras_de_acceso_a_servicios_de_salud_en_cabecera,Barreras de acceso a servicios de salud en Cabecera,Pobreza,Privaciones por hogar del IPM,Porcentaje (el valor esta multiplicado por 100),DANE - Encuesta Nacional de Calidad de Vida (ECV),33,2018,2024,7,"List(2018, 2019, 2020, 2021, 2022, 2023, 2024)",1.0,false
componente_de_nbi_por_dependencia_economica,Componente de NBI por dependencia economica,Pobreza,Necesidades Básicas Insatisfechas,Porcentaje (el valor esta multiplicado por 100),DANE - CNPV 2005 / DANE - CNPV 2018,33,2005,2018,2,"List(2005, 2018)",1.0,false
componente_de_nbi_por_dependencia_economica_en_cabecera,Componente de NBI por dependencia economica en Cabecera,Pobreza,Necesidades Básicas Insatisfechas,Porcentaje (el valor esta multiplicado por 100),DANE - CNPV 2005 / DANE - CNPV 2018,33,2005,2018,2,"List(2005, 2018)",1.0,false


## 3. Ministerio de Educación (variable objetivo)

In [ ]:
raw_edu, batch_edu = lote_vigente("educacion")
n_raw_edu = raw_edu.count()
edu = aplicar_contrato(raw_edu, "educacion")

METRICAS_EDU = [c for c in CONTRATOS["educacion"]["columnas"] if c not in ("anio", "cod_dpto", "departamento_nombre")]
TASAS_EDU = [c for c in METRICAS_EDU if c not in ("poblacion_5_16", "tamano_promedio_de_grupo")]
ESCALA_DECIMAL = {"poblacion_5_16": ","}   # conteos: el punto es separador de miles

edu_p = cachear(edu.select(
    parsear_entero("anio").alias("anio"),
    parsear_entero("cod_dpto").alias("cod_dpto"),
    homologar_nombre_departamento("departamento_nombre").alias("departamento_reportado"),
    *[parsear_numero(c, decimal=ESCALA_DECIMAL.get(c, "auto")).alias(c) for c in METRICAS_EDU],
    *[F.col(c).cast("string").contains("%").alias(f"_pct_{c}") for c in TASAS_EDU],
    *[F.col(c).alias(f"_raw_{c}") for c in METRICAS_EDU],
    "_corrupt_record", "_batch_id",
))

fallas = contar_fallas_parseo(edu_p, [(f"_raw_{c}", c) for c in METRICAS_EDU])
verificar("plata", "educacion", "valores no parseables (total columnas)", sum(fallas.values()) == 0,
          valor=sum(fallas.values()), umbral=0, dimension="validez",
          detalle=str({k: v for k, v in fallas.items() if v}))

# Escala: si una columna de tasa no trae '%' y su máximo es ≤ 1, viene como fracción → se lleva a 0-100
agg = edu_p.select(*[F.max(c).alias(f"max_{c}") for c in TASAS_EDU],
                   *[F.max(F.col(f"_pct_{c}").cast("int")).alias(f"pct_{c}") for c in TASAS_EDU]).first()
reescaladas = [c for c in TASAS_EDU if (agg[f"pct_{c}"] or 0) == 0 and agg[f"max_{c}"] is not None and agg[f"max_{c}"] <= 1.0]
for c in reescaladas:
    edu_p = edu_p.withColumn(c, F.col(c) * 100)
verificar("plata", "educacion", "columnas de tasa reescaladas de fracción a porcentaje", True,
          valor=len(reescaladas), dimension="conformidad", detalle=str(reescaladas))

edu_p = edu_p.join(dim_dpto.select("cod_dpto", F.col("nombre_canonico").alias("_nombre_dim"),
                                   F.col("departamento").alias("departamento")), "cod_dpto", "left")
motivo_e = (F.when(F.col("_corrupt_record").isNotNull(), "registro_corrupto")
            .when(F.col("anio").isNull(), "anio_nulo")
            .when(F.col("_nombre_dim").isNull(), "departamento_no_valido"))
edu_p = edu_p.withColumn("motivo_cuarentena", motivo_e)

cuarentena_edu = edu_p.filter(F.col("motivo_cuarentena").isNotNull())
verificar("plata", "educacion", "registros en cuarentena", cuarentena_edu.count() == 0,
          valor=cuarentena_edu.count(), umbral=0, dimension="validez")
guardar_tabla(cuarentena_edu.select("motivo_cuarentena", "anio", "cod_dpto", "departamento_reportado", "_batch_id"),
              SCHEMA_SILVER, "educacion_cuarentena")

edu_v = edu_p.filter(F.col("motivo_cuarentena").isNull() & F.col("anio").between(ANIO_MIN_EDUCACION, ANIO_MAX))
inc_e = edu_v.filter(F.col("departamento_reportado").isNotNull() & (F.col("departamento_reportado") != F.col("_nombre_dim"))).count()
verificar("plata", "educacion", "consistencia código DIVIPOLA ↔ nombre", inc_e == 0, valor=inc_e, umbral=0,
          dimension="consistencia")
verificar_unicidad(edu_v, ["cod_dpto", "anio"], "plata", "educacion_departamento_anio")

panel_edu = edu_v.filter(F.col("anio").between(ANIO_MIN, ANIO_MAX)).count()
esperado = len(CODIGOS_DEPARTAMENTO) * (ANIO_MAX - ANIO_MIN + 1)
verificar("plata", "educacion", "panel completo 2018-2024 (33 × 7)", panel_edu == esperado, valor=panel_edu,
          umbral=esperado, dimension="completitud")

n_obj_nulo = edu_v.filter(F.col("anio").between(ANIO_MIN, ANIO_MAX) & F.col("tasa_desercion").isNull()).count()
verificar("plata", "educacion", "variable objetivo nula en 2018-2024", n_obj_nulo == 0, valor=n_obj_nulo, umbral=0,
          dimension="completitud", detalle="la variable objetivo nunca se imputa")
fuera_obj = edu_v.filter(F.col("tasa_desercion").isNotNull() & ~F.col("tasa_desercion").between(0, 100)).count()
verificar("plata", "educacion", "tasa_desercion dentro de [0, 100]", fuera_obj == 0, valor=fuera_obj, umbral=0,
          severidad="BLOQUEANTE", dimension="validez")

✅ [plata/educacion] existe lote vigente en Bronce: valor=None umbral=None 
educacion: lote vigente educacion-20261005T014216-7ce4f8 (Educacion.csv, 462 filas)
✅ [plata/educacion] contrato: columnas obligatorias presentes: valor=0 umbral=0 faltantes=[]; columnas disponibles=['_archivo_modificacion', '_archivo_origen', '_batch_id', '_corrupt_record', '_ejecucion_id', '_fecha_ingesta', '_fuente', '_hash_registro', 'ano', 'aprobacion', 'aprobacion_media', 'aprobacion_primaria', 'aprobacion_secundaria', 'aprobacion_transicion', 'cobertura_bruta', 'cobertura_bruta_media', 'cobertura_bruta_primaria', 'cobertura_bruta_secundaria', 'cobertura_bruta_transicion', 'cobertura_neta', 'cobertura_neta_media', 'cobertura_neta_primaria', 'cobertura_neta_secundaria', 'cobertura_neta_transicion', 'codigo_departamento', 'departamento', 'desercion', 'desercion_media', 'desercion_primaria', 'desercion_secundaria', 'desercion_transicion', 'poblacion_5_16', 'repitencia', 'repitencia_media', 'repitencia_primari

True

### Coherencia interna de los indicadores del MEN y detección de columnas desalineadas
Para el sector oficial, aprobación + reprobación + deserción intra-anual ≈ 100 % (la diferencia son traslados).
Una columna se marca **no confiable** (y se excluye del modelado, quedando disponible para auditoría) si: (i) su mediana es
inverosímil para su tipo (aprobación < 50 %, o deserción/reprobación/repitencia > 30 %); (ii) es idéntica a otra columna en
más de la mitad de las filas y es la inverosímil del par (si ambas son verosímiles se descartan las dos); o (iii) la identidad
del nivel no se cumple en ≥ 80 % de las filas (excepto la deserción, que es la variable de interés). En la muestra de Bronce se observa, por ejemplo, `aprobacion_secundaria = desercion_secundaria` y
`reprobacion_primaria = aprobacion_primaria`, señal de desalineación de columnas en el archivo publicado.

In [ ]:
TOLERANCIA_PP = 3.0
niveles = {"total": ("aprobacion", "reprobacion", "tasa_desercion")}
for nv in ["transicion", "primaria", "secundaria", "media"]:
    niveles[nv] = (f"aprobacion_{nv}", f"reprobacion_{nv}", f"desercion_{nv}")

coh_exprs = []
for nv, (a, r, d) in niveles.items():
    suma = F.col(a) + F.col(r) + F.col(d)
    coh_exprs.append(F.avg((F.abs(suma - 100) <= TOLERANCIA_PP).cast("double")).alias(f"coh_{nv}"))
coh = edu_v.select(*coh_exprs).first()

comparables = [c for c in TASAS_EDU if any(c.startswith(p) for p in ("aprobacion", "reprobacion", "desercion", "tasa_desercion", "repitencia"))]
pares = [(x, y) for i, x in enumerate(comparables) for y in comparables[i + 1:]]
ident = edu_v.select(*[F.avg((F.abs(F.col(x) - F.col(y)) < 1e-9).cast("double")).alias(f"{x}__{y}") for x, y in pares]).first()
pares_identicos = [(x, y, float(ident[f"{x}__{y}"])) for x, y in pares if (ident[f"{x}__{y}"] or 0) > 0.5]

# Rango plausible de la mediana según el tipo de indicador (en %)
def rango_plausible(c):
    if c.startswith("aprobacion"):
        return (50.0, 100.0)
    if c.startswith(("desercion", "tasa_desercion", "reprobacion", "repitencia")):
        return (0.0, 30.0)
    return None

medianas = edu_v.select(*[F.expr(f"percentile_approx({c}, 0.5)").alias(c) for c in comparables]).first().asDict()
plausible = {c: (rango_plausible(c) is None or medianas[c] is None
                 or rango_plausible(c)[0] <= medianas[c] <= rango_plausible(c)[1]) for c in comparables}

filas_calidad = []
for c in METRICAS_EDU:
    nivel = next((nv for nv, cols in niveles.items() if c in cols), None)
    pct_coh = float(coh[f"coh_{nivel}"]) if nivel and coh[f"coh_{nivel}"] is not None else None
    pares_c = [(y if x == c else x, p) for x, y, p in pares_identicos if c in (x, y)]
    # En un par idéntico se descarta la columna cuyo valor típico es inverosímil para su tipo
    # (p. ej. aprobación ≈ 4 % es una deserción desplazada). Si ambas son verosímiles, se descartan ambas.
    sospechosa = any((not plausible.get(c, True)) or plausible.get(otra, True) for otra, _ in pares_c)
    motivo = []
    if not plausible.get(c, True):
        motivo.append(f"mediana inverosímil ({medianas.get(c):.2f})")
    if pares_c and sospechosa:
        motivo.append("idéntica a " + ", ".join(f"{o} ({p:.0%})" for o, p in pares_c))
    if pct_coh is not None and pct_coh < 0.8 and not c.startswith(("desercion_", "tasa_desercion")):
        motivo.append(f"identidad aprobación+reprobación+deserción del nivel cumple solo en {pct_coh:.0%}")
    confiable = not motivo
    filas_calidad.append((c, nivel, pct_coh, medianas.get(c), "; ".join(motivo) or None, bool(confiable)))

calidad_cols = spark.createDataFrame(filas_calidad, "columna STRING, nivel STRING, pct_filas_coherentes DOUBLE, mediana DOUBLE, motivo_exclusion STRING, confiable BOOLEAN")
guardar_tabla(calidad_cols, SCHEMA_SILVER, "educacion_calidad_columnas")
display(calidad_cols)

verificar("plata", "educacion", "identidad aprobación+reprobación+deserción≈100 (total)", (coh["coh_total"] or 0) >= 0.8,
          valor=round(coh["coh_total"] or 0, 3), umbral=">=0.8", dimension="consistencia")
no_conf = [f[0] for f in filas_calidad if not f[5]]
verificar("plata", "educacion", "columnas no confiables (excluidas del modelado)", len(no_conf) == 0,
          valor=len(no_conf), umbral=0, dimension="consistencia", detalle=str(no_conf))

  ✔ desercion_escolar.capa_plata.educacion_calidad_columnas (overwrite) → 34 filas


columna,nivel,pct_filas_coherentes,mediana,motivo_exclusion,confiable
poblacion_5_16,null,null,null,null,true
tasa_matriculacion_5_16,null,null,null,null,true
cobertura_neta,null,null,null,null,true
cobertura_neta_transicion,null,null,null,null,true
cobertura_neta_primaria,null,null,null,null,true
cobertura_neta_secundaria,null,null,null,null,true
cobertura_neta_media,null,null,null,null,true
cobertura_bruta,null,null,null,null,true
cobertura_bruta_transicion,null,null,null,null,true
cobertura_bruta_primaria,null,null,null,null,true


✅ [plata/educacion] identidad aprobación+reprobación+deserción≈100 (total): valor=1.0 umbral=>=0.8 
⚠️ [plata/educacion] columnas no confiables (excluidas del modelado): valor=7 umbral=0 ['aprobacion_transicion', 'aprobacion_primaria', 'aprobacion_secundaria', 'reprobacion_transicion', 'reprobacion_primaria', 'reprobacion_secundaria', 'repitencia_transicion']


False

In [ ]:
educacion_anual = edu_v.select("cod_dpto", "departamento", "anio", *METRICAS_EDU,
                               F.lit(batch_edu).alias("_batch_id"))
guardar_tabla(educacion_anual, SCHEMA_SILVER, "educacion_departamento_anio", particion="anio",
              comentario="MEN por departamento-anio 2011-2024; tasas en escala 0-100; tasa_desercion = desercion intra-anual sector oficial")
liberar(edu_p)

  ✔ desercion_escolar.capa_plata.educacion_departamento_anio (overwrite) → 462 filas


## 4. Resumen de calidad de la capa Plata

In [ ]:
persistir_calidad()
display(leer_tabla(SCHEMA_GOB, "control_calidad").filter(F.col("ejecucion_id") == EJECUCION_ID)
        .filter("capa = 'plata'").orderBy("tabla", "regla"))

  ✔ desercion_escolar.gobierno_datos.control_calidad (append) → 176 filas


ejecucion_id,fecha_evaluacion,capa,tabla,regla,dimension_calidad,severidad,valor_observado,umbral,estado,detalle
20261005T153646-921cd2,2026-10-05T15:42:25.075Z,plata,educacion,columnas de tasa reescaladas de fracción a porcentaje,conformidad,ADVERTENCIA,0.0,null,OK,[]
20261005T153646-921cd2,2026-10-05T15:42:51.938Z,plata,educacion,columnas no confiables (excluidas del modelado),consistencia,ADVERTENCIA,7.0,0,ALERTA,"['aprobacion_transicion', 'aprobacion_primaria', 'aprobacion_secundaria', 'reprobacion_transicion', 'reprobacion_primaria', 'reprobacion_secundaria', 'repitencia_transicion']"
20261005T153646-921cd2,2026-10-05T15:42:33.205Z,plata,educacion,consistencia código DIVIPOLA ↔ nombre,consistencia,ADVERTENCIA,0.0,0,OK,
20261005T153646-921cd2,2026-10-05T15:42:20.097Z,plata,educacion,contrato: columnas obligatorias presentes,conformidad,BLOQUEANTE,0.0,0,OK,"faltantes=[]; columnas disponibles=['_archivo_modificacion', '_archivo_origen', '_batch_id', '_corrupt_record', '_ejecucion_id', '_fecha_ingesta', '_fuente', '_hash_registro', 'ano', 'aprobacion', 'aprobacion_media', 'aprobacion_primaria', 'aprobacion_secundaria', 'aprobacion_transicion', 'cobertura_bruta', 'cobertura_bruta_media', 'cobertura_bruta_primaria', 'cobertura_bruta_secundaria', 'cobertura_bruta_transicion', 'cobertura_neta', 'cobertura_neta_media', 'cobertura_neta_primaria', 'cobertura_neta_secundaria', 'cobertura_neta_transicion', 'codigo_departamento', 'departamento', 'desercion', 'desercion_media', 'desercion_primaria', 'desercion_secundaria', 'desercion_transicion', 'poblacion_5_16', 'repitencia', 'repitencia_media', 'repitencia_primaria', 'repitencia_secundaria', 'repitencia_transicion', 'reprobacion', 'reprobacion_media', 'reprobacion_primaria', 'reprobacion_secundaria', 'reprobacion_transicion', 'sedes_conectadas_a_internet', 'tamano_promedio_de_grupo', 'tasa_matriculacion_5_16']"
20261005T153646-921cd2,2026-10-05T15:42:20.097Z,plata,educacion,contrato: columnas opcionales presentes,conformidad,ADVERTENCIA,0.0,0,OK,ausentes (quedan nulas)=[]
20261005T153646-921cd2,2026-10-05T15:42:19.388Z,plata,educacion,existe lote vigente en Bronce,completitud,BLOQUEANTE,null,null,OK,
20261005T153646-921cd2,2026-10-05T15:42:51.938Z,plata,educacion,identidad aprobación+reprobación+deserción≈100 (total),consistencia,ADVERTENCIA,1.0,>=0.8,OK,
20261005T153646-921cd2,2026-10-05T15:42:37.197Z,plata,educacion,panel completo 2018-2024 (33 × 7),completitud,ADVERTENCIA,231.0,231,OK,
20261005T153646-921cd2,2026-10-05T15:42:28.026Z,plata,educacion,registros en cuarentena,validez,ADVERTENCIA,0.0,0,OK,
20261005T153646-921cd2,2026-10-05T15:42:40.587Z,plata,educacion,"tasa_desercion dentro de [0, 100]",validez,BLOQUEANTE,0.0,0,OK,


## 03_Capa_Oro

<!-- merged from: 03_Capa_Oro.ipynb (Jupy Tools) -->

# 03 · Capa Oro — *feature store* departamento-año, modelo dimensional y análisis estadístico

**CRISP-DM:** Preparación de datos (integración) + Comprensión de los datos (análisis) · **Medallion:** Oro

### Definición operacional del problema (CRISP-DM · Comprensión del negocio)
| Elemento | Definición |
|---|---|
| Unidad de análisis | Departamento (33, llave DIVIPOLA) × año, 2018–2024 (231 observaciones) |
| Variable objetivo | `tasa_desercion`: tasa de deserción **intra-anual del sector oficial** (MEN), en %, del año *t* |
| Decisión que habilita | Focalizar estrategias de permanencia (PAE, transporte, búsqueda activa, articulación con salud) en los departamentos con mayor deserción esperada |
| Usuario | Secretarías de educación departamentales, MEN, entes de control (Procuraduría) |
| Escenario **nowcast** (principal) | Estimar la tasa del año *t* al cierre epidemiológico de *t*, antes de la publicación oficial del MEN (*t+1*). Usa: SIVIGILA de *t* (reporte semanal), MEN hasta *t−1*, Terridata publicado hasta *t−1* |
| Escenario **pronóstico** (*t+1*) | Anticipar la tasa del año siguiente con información disponible al cierre de *t−1* (todo rezagado). Produce la proyección **2025** |
| Criterio de éxito | Superar a la línea base de persistencia (tasa del año anterior) en RMSE de validación temporal y en la prueba 2024 |
| Restricción ética | Análisis ecológico (agregado): no se infiere riesgo individual (falacia ecológica, Robinson 1950) ni causalidad |

### Disponibilidad temporal de las variables (corrige la posible fuga de información del informe)
Ninguna variable del MEN del mismo año *t* entra como predictor (se publica junto con la variable objetivo). Los
indicadores de Terridata se unen **as-of**: el último valor publicado con año ≤ *t−1*; los indicadores censales de un
solo año (CNPV 2018) se tratan como características estructurales invariantes y se documentan como tal.

In [ ]:
%run ./00_Config_Utilidades

Entorno Databricks: True | formato: delta | ejecución: 20261005T154822-f0a9f7


Utilidades cargadas ✔


In [ ]:
import numpy as np
import pandas as pd

crear_esquemas()
dim_dpto = df_departamentos()
edu = leer_tabla(SCHEMA_SILVER, "educacion_departamento_anio")
siv = leer_tabla(SCHEMA_SILVER, "sivigila_departamento_anio")
pob = leer_tabla(SCHEMA_SILVER, "pobreza_departamento_indicador")
cat_pob = leer_tabla(SCHEMA_SILVER, "pobreza_catalogo_indicadores")
cal_edu = leer_tabla(SCHEMA_SILVER, "educacion_calidad_columnas")

for nombre, df, llaves in [("educacion", edu, ["cod_dpto", "anio"]), ("sivigila", siv, ["cod_dpto", "anio"]),
                           ("pobreza", pob, ["cod_dpto", "indicador_id", "anio"])]:
    verificar_unicidad(df, llaves, "oro", f"entrada_{nombre}")

version_datos = "|".join(sorted({r["_batch_id"] for r in edu.select("_batch_id").distinct().collect()}
                                | {r["_batch_id"] for r in pob.select("_batch_id").distinct().collect()}
                                | {r["_batch_id"] for r in leer_tabla(SCHEMA_SILVER, "sivigila_limpio").select("_batch_id").distinct().collect()}))
print("Versión de datos:", version_datos)

✅ [oro/entrada_educacion] unicidad de la llave ('cod_dpto', 'anio'): valor=0 umbral=0 
✅ [oro/entrada_sivigila] unicidad de la llave ('cod_dpto', 'anio'): valor=0 umbral=0 
✅ [oro/entrada_pobreza] unicidad de la llave ('cod_dpto', 'indicador_id', 'anio'): valor=0 umbral=0 
Versión de datos: educacion-20261005T014216-7ce4f8|pobreza-20261005T014216-7ce4f8|sivigila-20261005T014216-7ce4f8


## 1. Dimensiones

In [ ]:
guardar_tabla(dim_dpto, SCHEMA_GOLD, "dim_departamento", comentario="Departamentos de Colombia con codigo DIVIPOLA, region y coordenadas")

anios = spark.range(ANIO_MIN_EDUCACION, ANIO_PRONOSTICO + 1).select(F.col("id").cast("int").alias("anio"))
dim_anio = (spark.range(ANIO_MIN_POBREZA, ANIO_PRONOSTICO + 1).select(F.col("id").cast("int").alias("anio"))
            .withColumn("periodo_pandemia", F.col("anio").isin(2020, 2021).cast("int"))
            .withColumn("etapa", F.when(F.col("anio") < 2020, "Prepandemia").when(F.col("anio") <= 2021, "Pandemia").otherwise("Pospandemia"))
            .withColumn("en_periodo_estudio", F.col("anio").between(ANIO_MIN, ANIO_MAX))
            .withColumn("es_pronostico", F.col("anio") > ANIO_MAX))
guardar_tabla(dim_anio, SCHEMA_GOLD, "dim_anio")

guardar_tabla(leer_tabla(SCHEMA_SILVER, "sivigila_catalogo_eventos"), SCHEMA_GOLD, "dim_evento_sivigila")

DESCRIPCION_PERFIL = {
    "NUTRICIONAL": "Desnutrición aguda, bajo peso al nacer y mortalidad por desnutrición",
    "ITS": "Infecciones de transmisión sexual (sífilis, VIH, hepatitis B/C)",
    "VECTORIAL": "Dengue, malaria, zika, chikunguña, leishmaniasis, Chagas",
    "RESPIRATORIA": "IRA/IRAG, tuberculosis, COVID-19, tos ferina",
    "HIDRICA_ALIMENTARIA": "EDA, ETA, hepatitis A, fiebre tifoidea",
    "INMUNOPREVENIBLE": "Varicela, sarampión, rubéola, parotiditis, ESAVI",
    "ZOONOSIS_OTRAS_INFECCIOSAS": "Agresiones por animales transmisores de rabia, leptospirosis y otras infecciones",
    "VIOLENCIA": "Violencia de género e intrafamiliar, ataques con agentes químicos",
    "SALUD_MENTAL": "Intento de suicidio y consumo de sustancias",
    "MATERNO_PERINATAL": "Morbilidad materna extrema, mortalidad perinatal, defectos congénitos",
    "LESIONES_INTOXICACIONES": "Intoxicaciones, accidente ofídico, lesiones por pólvora",
    "CRONICAS_CANCER": "Cáncer en menores de 18 años y enfermedades huérfanas",
    "OTROS": "Eventos no clasificados en los perfiles anteriores",
}
dim_perfil = spark.createDataFrame(
    [(p, g, DESCRIPCION_PERFIL.get(p, "")) for p, g, _ in REGLAS_PERFIL_EVENTO] + [("OTROS", "OTROS", DESCRIPCION_PERFIL["OTROS"])],
    "perfil_evento STRING, grupo_perfil STRING, descripcion STRING")
guardar_tabla(dim_perfil, SCHEMA_GOLD, "dim_perfil_epidemiologico")

  ✔ desercion_escolar.capa_oro.dim_departamento (overwrite) → 33 filas
  ✔ desercion_escolar.capa_oro.dim_anio (overwrite) → 21 filas
  ✔ desercion_escolar.capa_oro.dim_evento_sivigila (overwrite) → 82 filas
  ✔ desercion_escolar.capa_oro.dim_perfil_epidemiologico (overwrite) → 13 filas


13

## 2. Variables educativas (solo rezagos: disponibles al cierre de *t−1*)

In [ ]:
confiables = {r["columna"] for r in cal_edu.filter("confiable").collect()}
COLS_EDU_REZAGO = [c for c in ["tasa_desercion", "desercion_primaria", "desercion_secundaria", "desercion_media",
                               "reprobacion", "repitencia", "aprobacion", "cobertura_neta", "cobertura_bruta",
                               "tasa_matriculacion_5_16", "tamano_promedio_de_grupo", "sedes_conectadas_a_internet"]
                   if c in confiables or c == "tasa_desercion"]
print("Columnas MEN confiables usadas como rezago:", COLS_EDU_REZAGO)

panel_edu = (dim_dpto.select("cod_dpto").crossJoin(anios)
             .join(edu.select("cod_dpto", "anio", "poblacion_5_16", *COLS_EDU_REZAGO), ["cod_dpto", "anio"], "left"))
if {"desercion_secundaria", "desercion_primaria"} <= set(COLS_EDU_REZAGO):
    panel_edu = panel_edu.withColumn("brecha_secundaria_primaria", F.col("desercion_secundaria") - F.col("desercion_primaria"))
    COLS_EDU_REZAGO.append("brecha_secundaria_primaria")

feat_edu = agregar_rezagos(panel_edu, COLS_EDU_REZAGO, 1)
feat_edu = agregar_rezagos(feat_edu, ["tasa_desercion"], 2)
feat_edu = agregar_rezagos(feat_edu, ["tasa_desercion"], 3)
feat_edu = agregar_rezagos(feat_edu, ["poblacion_5_16"], 1)
feat_edu = (feat_edu
            # La población 5-16 del año t es una proyección DANE conocida de antemano (no es fuga)
            .withColumn("poblacion_5_16", F.coalesce(F.col("poblacion_5_16"), F.col("poblacion_5_16_lag1")))
            .withColumn("tasa_desercion_media3", (F.col("tasa_desercion_lag1") + F.col("tasa_desercion_lag2") + F.col("tasa_desercion_lag3")) / 3)
            .withColumn("tasa_desercion_tendencia", F.col("tasa_desercion_lag1") - F.col("tasa_desercion_lag2")))

COLS_EDU_FEATURES = [f"{c}_lag1" for c in COLS_EDU_REZAGO] + ["tasa_desercion_lag2", "tasa_desercion_lag3",
                                                               "tasa_desercion_media3", "tasa_desercion_tendencia"]
feat_edu = feat_edu.select("cod_dpto", "anio", "tasa_desercion",
                           *[c for c in ["desercion_primaria", "desercion_secundaria", "desercion_media", "brecha_secundaria_primaria"] if c in feat_edu.columns],
                           "poblacion_5_16",
                           *[F.col(c).alias(f"edu_{c}") for c in COLS_EDU_FEATURES])

Columnas MEN confiables usadas como rezago: ['tasa_desercion', 'desercion_primaria', 'desercion_secundaria', 'desercion_media', 'reprobacion', 'repitencia', 'aprobacion', 'cobertura_neta', 'cobertura_bruta', 'tasa_matriculacion_5_16', 'tamano_promedio_de_grupo', 'sedes_conectadas_a_internet']


## 3. Variables epidemiológicas (SIVIGILA): tasas por 1.000 habitantes de 5–16 años
Los conteos absolutos confunden carga de morbilidad con tamaño poblacional (Bogotá vs Vaupés). Se normalizan por la
población 5–16 (proyección DANE incluida en el archivo del MEN). Es un normalizador de escala, no una incidencia
epidemiológica en sentido estricto (los eventos incluyen todas las edades): se documenta como limitación.

In [ ]:
cols_grupo = [f"casos_grupo_{g.lower()}" for g in GRUPOS_PERFIL]
cols_perfil = [f"casos_perfil_{p.lower()}" for p in PERFILES]
pob_denom = feat_edu.select("cod_dpto", "anio", "poblacion_5_16")

siv_t = (siv.join(pob_denom, ["cod_dpto", "anio"], "left")
         .select("cod_dpto", "anio", "sivigila_reporto", "tipos_evento", "semanas_con_reporte",
                 *[(F.col(c) / F.col("poblacion_5_16") * 1000).alias(c.replace("casos_", "tasa_")) for c in ["casos_total"] + cols_grupo + cols_perfil]))
COLS_SIV = [c for c in siv_t.columns if c.startswith("tasa_")] + ["tipos_evento"]
siv_t = siv_t.withColumn("proporcion_infecciosa_nutricional",
                         F.col("tasa_grupo_infecciosa") / (F.col("tasa_grupo_infecciosa") + F.col("tasa_grupo_nutricional")))
COLS_SIV.append("proporcion_infecciosa_nutricional")

siv_feat = agregar_rezagos(siv_t.select("cod_dpto", "anio", *COLS_SIV), COLS_SIV, 1)
siv_feat = siv_feat.select("cod_dpto", "anio", *[F.col(c).alias(f"siv_{c}") for c in COLS_SIV],
                           *[F.col(f"{c}_lag1").alias(f"siv_{c}_lag1") for c in COLS_SIV])
# Para 2025 solo existen los rezagos (SIVIGILA 2024)
siv_lag_2025 = (siv_t.filter(F.col("anio") == ANIO_MAX)
                .select("cod_dpto", F.lit(ANIO_PRONOSTICO).alias("anio"), *[F.col(c).alias(f"siv_{c}_lag1") for c in COLS_SIV]))
siv_feat = siv_feat.unionByName(siv_lag_2025, allowMissingColumns=True)

## 4. Variables socioeconómicas (Terridata) con unión *as-of* y selección explícita
Selección por cobertura (≥ 90 % de los departamentos y al menos un valor publicado antes de 2024). La lista resultante
es **explícita y persistida** (`dim_indicador_pobreza.seleccionado`), lo que fija el esquema de Oro entre ejecuciones.

In [ ]:
MIN_COBERTURA = 0.9
seleccion = (cat_pob.filter((F.col("pct_cobertura_departamentos") >= MIN_COBERTURA) & (F.col("anio_min") <= ANIO_MAX - 1))
             .orderBy("indicador_id"))
INDICADORES_SEL = [r["indicador_id"] for r in seleccion.collect()]
ESTRUCTURALES = {r["indicador_id"] for r in seleccion.filter("es_estructural").collect()}
verificar("oro", "pobreza", "indicadores Terridata seleccionados (cobertura ≥ 90%)", len(INDICADORES_SEL) > 0,
          valor=len(INDICADORES_SEL), umbral=">0", severidad="BLOQUEANTE", dimension="completitud")

guardar_tabla(cat_pob.withColumn("seleccionado", F.col("indicador_id").isin(INDICADORES_SEL))
              .withColumn("columna_feature", F.concat(F.lit("pob_"), F.col("indicador_id"))),
              SCHEMA_GOLD, "dim_indicador_pobreza")

anios_obj = spark.range(ANIO_MIN, ANIO_PRONOSTICO + 1).select(F.col("id").cast("int").alias("anio_objetivo"))
pob_sel = pob.filter(F.col("indicador_id").isin(INDICADORES_SEL) & F.col("valor").isNotNull())
asof = (pob_sel.crossJoin(anios_obj)
        .filter((F.col("anio") <= F.col("anio_objetivo") - 1) | F.col("indicador_id").isin(list(ESTRUCTURALES)))
        .withColumn("_r", F.row_number().over(Window.partitionBy("cod_dpto", "indicador_id", "anio_objetivo")
                                                .orderBy(F.desc("anio"))))
        .filter("_r = 1"))
pob_feat = (asof.groupBy("cod_dpto", F.col("anio_objetivo").alias("anio"))
            .pivot("indicador_id", INDICADORES_SEL)          # lista explícita → esquema estable
            .agg(F.first("valor")))
for c in INDICADORES_SEL:
    pob_feat = pob_feat.withColumnRenamed(c, f"pob_{c}")
antig = (asof.groupBy("cod_dpto", F.col("anio_objetivo").alias("anio"))
         .agg(F.avg(F.col("anio_objetivo") - F.col("anio")).alias("pob_antiguedad_media_anios")))
pob_feat = pob_feat.join(antig, ["cod_dpto", "anio"], "left")

✅ [oro/pobreza] indicadores Terridata seleccionados (cobertura ≥ 90%): valor=84 umbral=>0 
  ✔ desercion_escolar.capa_oro.dim_indicador_pobreza (overwrite) → 132 filas


## 5. Integración: *feature store* departamento-año (2018–2025)

In [ ]:
panel = dim_dpto.select("cod_dpto", "departamento", "region").crossJoin(
    spark.range(ANIO_MIN, ANIO_PRONOSTICO + 1).select(F.col("id").cast("int").alias("anio")))
n_panel = panel.count()

fs = (panel
      .join(feat_edu, ["cod_dpto", "anio"], "left")
      .join(siv_feat, ["cod_dpto", "anio"], "left")
      .join(pob_feat, ["cod_dpto", "anio"], "left")
      .withColumn("periodo_pandemia", F.col("anio").isin(2020, 2021).cast("int"))
      .withColumn("es_anio_pronostico", F.col("anio") > ANIO_MAX)
      .withColumn("_version_datos", F.lit(version_datos))
      .withColumn("_ejecucion_id", F.lit(EJECUCION_ID)))

verificar("oro", "feature_store", "cardinalidad: filas = 33 departamentos × años (sin fan-out)", fs.count() == n_panel,
          valor=fs.count(), umbral=n_panel, severidad="BLOQUEANTE", dimension="integridad")
verificar_unicidad(fs, ["cod_dpto", "anio"], "oro", "feature_store")

hist = fs.filter(~F.col("es_anio_pronostico"))
n_hist = hist.count()
cobertura = hist.select(
    F.avg(F.col("tasa_desercion").isNotNull().cast("double")).alias("objetivo"),
    F.avg(F.col("edu_tasa_desercion_lag1").isNotNull().cast("double")).alias("rezago_objetivo"),
    F.avg(F.col("siv_tasa_total").isNotNull().cast("double")).alias("sivigila"),
    F.avg(F.coalesce(*[F.col(f"pob_{c}") for c in INDICADORES_SEL]).isNotNull().cast("double")).alias("pobreza"),
).first().asDict()
for fuente_c, valor in cobertura.items():
    verificar("oro", "feature_store", f"cobertura del join: {fuente_c}", valor >= 0.95, valor=round(valor, 4),
              umbral=">=0.95", severidad="BLOQUEANTE" if fuente_c == "objetivo" else "ADVERTENCIA",
              dimension="completitud")

nulos_fs = perfil_nulos(hist)
guardar_tabla(spark.createDataFrame([(EJECUCION_ID, c, float(v)) for c, v in nulos_fs.items()],
                                    "ejecucion_id STRING, columna STRING, pct_nulos DOUBLE"),
              SCHEMA_GOB, "perfil_nulos_feature_store", modo="append")

guardar_tabla(fs, SCHEMA_GOLD, "feature_store_desercion", particion="anio",
              comentario="Tabla maestra analitica departamento-anio 2018-2025 con disponibilidad temporal controlada")

✅ [oro/feature_store] cardinalidad: filas = 33 departamentos × años (sin fan-out): valor=264 umbral=264 
✅ [oro/feature_store] unicidad de la llave ('cod_dpto', 'anio'): valor=0 umbral=0 
✅ [oro/feature_store] cobertura del join: objetivo: valor=1.0 umbral=>=0.95 
✅ [oro/feature_store] cobertura del join: rezago_objetivo: valor=1.0 umbral=>=0.95 
✅ [oro/feature_store] cobertura del join: sivigila: valor=1.0 umbral=>=0.95 
✅ [oro/feature_store] cobertura del join: pobreza: valor=1.0 umbral=>=0.95 
  ✔ desercion_escolar.gobierno_datos.perfil_nulos_feature_store (append) → 632 filas
  ✔ desercion_escolar.capa_oro.feature_store_desercion (overwrite) → 264 filas


264

### Diccionario de variables (fuente, disponibilidad y escenario de uso)

In [ ]:
def describir(col):
    if col.startswith("edu_"):
        return ("MEN", "Educativa (rezago)", "Cierre de t−1", "nowcast, pronóstico")
    if col.startswith("siv_") and col.endswith("_lag1"):
        return ("SIVIGILA", "Epidemiológica (rezago)", "Cierre de t−1", "pronóstico")
    if col.startswith("siv_"):
        return ("SIVIGILA", "Epidemiológica (año t)", "Semanal durante t", "nowcast")
    if col.startswith("pob_"):
        ind = col[4:]
        tipo = "Estructural censal (invariante)" if ind in ESTRUCTURALES else "As-of ≤ t−1"
        return ("Terridata (DNP)", "Socioeconómica", tipo, "nowcast, pronóstico")
    if col == "periodo_pandemia":
        return ("Calendario", "Contexto", "Conocida", "solo descriptiva (no predictor)")
    return ("MEN", "Objetivo / contexto", "Publicada en t+1", "no es predictor")

dicc = [(c, *describir(c)) for c in fs.columns if not c.startswith("_") and c not in ("cod_dpto", "departamento", "region", "anio", "es_anio_pronostico")]
guardar_tabla(spark.createDataFrame(dicc, "variable STRING, fuente STRING, grupo STRING, disponibilidad STRING, escenario_uso STRING"),
              SCHEMA_GOLD, "diccionario_variables")

  ✔ desercion_escolar.capa_oro.diccionario_variables (overwrite) → 151 filas


151

## 6. Objetivo específico 2 — correlaciones con la deserción
Para cada variable candidata se reportan: Pearson y Spearman (con p-valor), la correlación **intra-departamental**
(variables centradas por departamento: elimina diferencias estructurales fijas y es más defendible frente a confusión),
y el **q-valor de Benjamini-Hochberg** (control de comparaciones múltiples). Se evalúan tres objetivos: la tasa total,
la deserción en secundaria y la brecha secundaria − primaria (transición primaria→secundaria).

*Justificación del uso de pandas:* el conjunto analítico tiene ≤ 231 filas; scipy aporta las pruebas de hipótesis.

In [ ]:
from scipy import stats

def bh_qvalores(p):
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    m = ok.sum()
    if m == 0:
        return q
    orden = np.argsort(p[ok])
    pv = p[ok][orden]
    qv = np.minimum.accumulate((pv * m / np.arange(1, m + 1))[::-1])[::-1]
    tmp = np.empty(m); tmp[orden] = np.minimum(qv, 1)
    q[ok] = tmp
    return q


pdf = hist.toPandas()
OBJETIVOS = [c for c in ["tasa_desercion", "desercion_secundaria", "brecha_secundaria_primaria"] if c in pdf.columns]
candidatas = [c for c in pdf.columns if (c.startswith("siv_") or c.startswith("pob_") or c.startswith("edu_") or c == "periodo_pandemia")
              and pd.api.types.is_numeric_dtype(pdf[c])]
filas = []
for obj in OBJETIVOS:
    bloque = []
    for v in candidatas:
        d = pdf[["cod_dpto", obj, v]].dropna()
        if len(d) < 20 or d[v].nunique() < 3:
            continue
        r, p_r = stats.pearsonr(d[v], d[obj])
        rho, p_s = stats.spearmanr(d[v], d[obj])
        dm = d.assign(**{c: d[c] - d.groupby("cod_dpto")[c].transform("mean") for c in (v, obj)})
        g = d["cod_dpto"].nunique()
        if dm[v].std() > 1e-9 and dm[obj].std() > 1e-9:   # variables invariantes dentro del departamento → no estimable
            r_w = float(np.corrcoef(dm[v], dm[obj])[0, 1])
            gl = len(d) - g - 1
            t_w = r_w * np.sqrt(gl / max(1 - r_w ** 2, 1e-12))
            p_w = float(2 * stats.t.sf(abs(t_w), gl)) if gl > 0 else np.nan
        else:
            r_w, p_w = np.nan, np.nan
        fuente_v, grupo_v, disp, _ = describir(v)
        bloque.append([obj, v, fuente_v, grupo_v, len(d), float(r), float(p_r), float(rho), float(p_s), r_w, p_w])
    if bloque:
        q_s = bh_qvalores([b[8] for b in bloque])
        q_w = bh_qvalores([b[10] for b in bloque])
        for b, qs, qw in zip(bloque, q_s, q_w):
            filas.append(b + [float(qs), float(qw)])

corr = pd.DataFrame(filas, columns=["objetivo", "variable", "fuente", "grupo", "n", "pearson_r", "pearson_p",
                                    "spearman_rho", "spearman_p", "r_intra_departamental", "p_intra_departamental",
                                    "q_bh_spearman", "q_bh_intra"])
corr["abs_spearman"] = corr["spearman_rho"].abs()
corr["significativa_fdr_5pct"] = corr["q_bh_spearman"] < 0.05
corr["ranking"] = corr.groupby("objetivo")["abs_spearman"].rank(ascending=False, method="first").astype(int)
guardar_tabla(a_spark(corr), SCHEMA_GOLD, "analisis_correlaciones",
              comentario="Objetivo especifico 2: correlaciones Pearson/Spearman/intra-departamental con FDR")
display(spark.table(fqtn(SCHEMA_GOLD, "analisis_correlaciones")).filter("fuente = 'SIVIGILA'").orderBy("objetivo", "ranking"))

  ✔ desercion_escolar.capa_oro.analisis_correlaciones (overwrite) → 426 filas


objetivo,variable,fuente,grupo,n,pearson_r,pearson_p,spearman_rho,spearman_p,r_intra_departamental,p_intra_departamental,q_bh_spearman,q_bh_intra,abs_spearman,significativa_fdr_5pct,ranking
brecha_secundaria_primaria,siv_tasa_total,SIVIGILA,Epidemiológica (año t),231,0.29275446477938155,6.04314819270886E-6,0.4870391590292204,3.661775556435842E-15,0.20913927404977425,0.0030313489618203696,8.666202150231492E-14,0.012038785876943753,0.4870391590292204,true,7
brecha_secundaria_primaria,siv_tasa_grupo_psicosocial,SIVIGILA,Epidemiológica (año t),231,0.318287343903062,7.796688052355218E-7,0.4717841528027971,3.322981701361592E-14,0.2895224666133101,3.3587704434740034E-5,5.898292519916826E-13,3.5913007049452806E-4,0.4717841528027971,true,8
brecha_secundaria_primaria,siv_tasa_perfil_salud_mental,SIVIGILA,Epidemiológica (año t),231,0.318287343903062,7.796688052355218E-7,0.4717841528027971,3.322981701361592E-14,0.2895224666133101,3.3587704434740034E-5,5.898292519916826E-13,3.5913007049452806E-4,0.4717841528027971,true,9
brecha_secundaria_primaria,siv_tasa_grupo_infecciosa,SIVIGILA,Epidemiológica (año t),231,0.26070646541575176,6.0597072400773185E-5,0.4600850296767925,1.6769528876347892E-13,0.1828861818520959,0.009723275423282161,2.381273100441401E-12,0.03378838209590551,0.4600850296767925,true,12
brecha_secundaria_primaria,siv_tasa_perfil_lesiones_intoxicaciones,SIVIGILA,Epidemiológica (año t),231,0.4052924448919931,1.5195718803599731E-10,0.4546020088358456,3.5065784902997E-13,0.2969734186285231,2.0495338296774596E-5,4.526674051114158E-12,2.589865475683335E-4,0.4546020088358456,true,13
brecha_secundaria_primaria,siv_tasa_grupo_psicosocial_lag1,SIVIGILA,Epidemiológica (rezago),198,0.2871022863704863,4.114032803215722E-5,0.431393374150014,2.2244192596605618E-10,0.1129100013630307,0.147508870064805,1.6624607098515777E-9,0.2500455236464378,0.431393374150014,true,15
brecha_secundaria_primaria,siv_tasa_perfil_salud_mental_lag1,SIVIGILA,Epidemiológica (rezago),198,0.2871022863704863,4.114032803215722E-5,0.431393374150014,2.2244192596605618E-10,0.1129100013630307,0.147508870064805,1.6624607098515777E-9,0.2500455236464378,0.431393374150014,true,16
brecha_secundaria_primaria,siv_tasa_grupo_materno_perinatal_lag1,SIVIGILA,Epidemiológica (rezago),198,0.2825558393011486,5.501428198854222E-5,0.42748748346686,3.3576534540553813E-10,0.16602441904671664,0.03253536787046258,2.2704132879803056E-9,0.07413796940974261,0.42748748346686,true,17
brecha_secundaria_primaria,siv_tasa_perfil_materno_perinatal_lag1,SIVIGILA,Epidemiológica (rezago),198,0.2825558393011486,5.501428198854222E-5,0.42748748346686,3.3576534540553813E-10,0.16602441904671664,0.03253536787046258,2.2704132879803056E-9,0.07413796940974261,0.42748748346686,true,18
brecha_secundaria_primaria,siv_tasa_perfil_vectorial,SIVIGILA,Epidemiológica (año t),231,0.2387407928104966,2.502964764389866E-4,0.4271348198273135,1.1645945055652704E-11,0.13889528880998664,0.0504051339723713,1.2720955368482183E-10,0.10947365034624391,0.4271348198273135,true,19


### Picos históricos de deserción en secundaria vs. carga epidemiológica
Para cada departamento se identifica el año de mayor deserción en secundaria y se mide cuántas desviaciones estándar
(respecto a su propia historia) se separa cada tasa epidemiológica ese año. Un z medio positivo y consistente indica
que los picos de deserción coinciden con picos de morbilidad.

In [ ]:
obj_pico = "desercion_secundaria" if "desercion_secundaria" in pdf.columns else "tasa_desercion"
vars_pico = [c for c in pdf.columns if c.startswith("siv_tasa_grupo_") and not c.endswith("_lag1")] + ["siv_tasa_total"]
reg = []
for cod, g in pdf.dropna(subset=[obj_pico]).groupby("cod_dpto"):
    if len(g) < 4:
        continue
    fila_pico = g.loc[g[obj_pico].idxmax()]
    for v in vars_pico:
        s = g[v].dropna()
        z = (fila_pico[v] - s.mean()) / s.std() if s.std() and not pd.isna(fila_pico[v]) else np.nan
        reg.append([int(cod), fila_pico["departamento"], int(fila_pico["anio"]), obj_pico, float(fila_pico[obj_pico]), v, float(z) if not pd.isna(z) else None])
picos = pd.DataFrame(reg, columns=["cod_dpto", "departamento", "anio_pico", "indicador_desercion", "valor_pico", "variable_epidemiologica", "z_en_anio_pico"])
picos["z_en_anio_pico"] = pd.to_numeric(picos["z_en_anio_pico"], errors="coerce").astype(float)
guardar_tabla(a_spark(picos), SCHEMA_GOLD, "analisis_picos_desercion")
resumen_picos = picos.groupby("variable_epidemiologica")["z_en_anio_pico"].agg(["mean", "count"])
for v, r in resumen_picos.iterrows():
    serie = picos.loc[picos.variable_epidemiologica == v, "z_en_anio_pico"].dropna()
    p = stats.wilcoxon(serie).pvalue if len(serie) >= 10 and serie.abs().sum() > 0 else np.nan
    print(f"{v:40s} z medio en año pico = {r['mean']:+.2f} (n={int(r['count'])}, p Wilcoxon={p:.3f})")

  ✔ desercion_escolar.capa_oro.analisis_picos_desercion (overwrite) → 198 filas
siv_tasa_grupo_infecciosa                z medio en año pico = -0.22 (n=33, p Wilcoxon=0.028)
siv_tasa_grupo_materno_perinatal         z medio en año pico = +0.06 (n=33, p Wilcoxon=0.751)
siv_tasa_grupo_nutricional               z medio en año pico = +0.34 (n=33, p Wilcoxon=0.083)
siv_tasa_grupo_otros                     z medio en año pico = -0.23 (n=33, p Wilcoxon=0.083)
siv_tasa_grupo_psicosocial               z medio en año pico = -0.18 (n=33, p Wilcoxon=0.140)
siv_tasa_total                           z medio en año pico = -0.22 (n=33, p Wilcoxon=0.083)


## 7. Objetivo específico 4 — interacción perfil epidemiológico × pobreza
Regresión lineal con errores estándar **robustos por conglomerado (departamento)**, variables estandarizadas:

`deserción = β1·infecciosas + β2·nutricionales + β3·pobreza + β4·infecciosas×pobreza + β5·nutricionales×pobreza + efectos fijos`

* **Especificación A** (efectos fijos de año): compara departamentos entre sí.
* **Especificación B** (efectos fijos de departamento y año): usa solo la variación *dentro* de cada departamento; es
  la más exigente frente a confusión estructural.

Los coeficientes son asociaciones condicionales, no efectos causales.

In [ ]:
def elegir_pobreza(columnas):
    patrones = [r"^pob_indice_de_pobreza_multidimensional(_ipm)?$", r"^pob_.*pobreza_multidimensional.*", r"^pob_.*ipm.*",
                r"^pob_incidencia_de_la_pobreza_monetaria(?!.*extrema).*", r"^pob_.*pobreza.*"]
    for pat in patrones:
        m = [c for c in columnas if re.match(pat, c) and "cabecera" not in c and "rural" not in c and "centros" not in c]
        if m:
            return sorted(m, key=len)[0]
    m = [c for c in columnas if c.startswith("pob_") and c != "pob_antiguedad_media_anios"]
    return m[0] if m else None


VAR_POBREZA = elegir_pobreza(pdf.columns)
print("Indicador de pobreza para la interacción:", VAR_POBREZA)


def ols_cluster(y, X, grupos, nombres):
    X = np.asarray(X, float); y = np.asarray(y, float); grupos = np.asarray(grupos)
    n, k = X.shape
    xtx_inv = np.linalg.pinv(X.T @ X)
    beta = xtx_inv @ X.T @ y
    u = y - X @ beta
    carne = np.zeros((k, k))
    gs = np.unique(grupos)
    for g in gs:
        s = X[grupos == g].T @ u[grupos == g]
        carne += np.outer(s, s)
    G = len(gs)
    V = (G / (G - 1)) * ((n - 1) / max(n - k, 1)) * xtx_inv @ carne @ xtx_inv
    se = np.sqrt(np.clip(np.diag(V), 0, None))
    t = np.divide(beta, se, out=np.full_like(beta, np.nan), where=se > 0)
    p = 2 * stats.t.sf(np.abs(t), G - 1)
    r2 = 1 - (u @ u) / (((y - y.mean()) ** 2).sum())
    return pd.DataFrame({"termino": nombres, "coeficiente": beta, "error_estandar_cluster": se, "t": t, "p_valor": p,
                         "ic95_inf": beta - stats.t.ppf(0.975, G - 1) * se, "ic95_sup": beta + stats.t.ppf(0.975, G - 1) * se}), r2, n


resultados_int = []
if VAR_POBREZA:
    # En el análisis de asociación (no predictivo) la pobreza se toma contemporánea: último valor con año ≤ t
    # (los indicadores censales de un único año se asignan a todos los años).
    pob_larga = pob.filter((F.col("indicador_id") == VAR_POBREZA[4:]) & F.col("valor").isNotNull()).select("cod_dpto", "anio", "valor").toPandas()
    contemp = []
    for (cod, a) in pdf[["cod_dpto", "anio"]].itertuples(index=False):
        serie = pob_larga[(pob_larga.cod_dpto == cod)]
        previos = serie[serie.anio <= a]
        valor = previos.sort_values("anio").valor.iloc[-1] if len(previos) else (serie.valor.iloc[0] if serie.anio.nunique() == 1 else np.nan)
        contemp.append(valor)
    pdf["pobreza_contemporanea"] = contemp
    d = pdf[["cod_dpto", "anio", "tasa_desercion", "siv_tasa_grupo_infecciosa", "siv_tasa_grupo_nutricional", "pobreza_contemporanea"]].dropna().copy()
    d = d.rename(columns={"pobreza_contemporanea": VAR_POBREZA})
    z = lambda s: (s - s.mean()) / s.std()
    d["infecciosas"] = z(np.log1p(d["siv_tasa_grupo_infecciosa"]))
    d["nutricionales"] = z(np.log1p(d["siv_tasa_grupo_nutricional"]))
    d["pobreza"] = z(d[VAR_POBREZA])
    d["infecciosas_x_pobreza"] = d["infecciosas"] * d["pobreza"]
    d["nutricionales_x_pobreza"] = d["nutricionales"] * d["pobreza"]
    base_terms = ["infecciosas", "nutricionales", "pobreza", "infecciosas_x_pobreza", "nutricionales_x_pobreza"]
    fe_anio = pd.get_dummies(d["anio"], prefix="anio", drop_first=True, dtype=float)
    fe_dpto = pd.get_dummies(d["cod_dpto"], prefix="dpto", drop_first=True, dtype=float)
    for espec, extras in [("A: efectos fijos de año", [fe_anio]), ("B: efectos fijos de departamento y año", [fe_dpto, fe_anio])]:
        X = pd.concat([pd.Series(1.0, index=d.index, name="intercepto"), d[base_terms]] + extras, axis=1)
        tabla, r2, n = ols_cluster(d["tasa_desercion"].values, X.values, d["cod_dpto"].values, list(X.columns))
        tabla = tabla[tabla.termino.isin(["intercepto"] + base_terms)].assign(especificacion=espec, r2=r2, n=n,
                                                                              variable_pobreza=VAR_POBREZA)
        resultados_int.append(tabla)
    interaccion = pd.concat(resultados_int, ignore_index=True)
    guardar_tabla(a_spark(interaccion), SCHEMA_GOLD, "analisis_interaccion",
                  comentario="Objetivo especifico 4: OLS con errores robustos por departamento")
    display(spark.table(fqtn(SCHEMA_GOLD, "analisis_interaccion")))

    # Cuadrantes descriptivos (para la matriz del tablero)
    d["perfil_epidemiologico"] = np.where(d["infecciosas"] >= d["nutricionales"], "Predominio infeccioso", "Predominio nutricional")
    d["nivel_pobreza"] = np.where(d["pobreza"] >= 0, "Pobreza alta", "Pobreza baja")
    d["carga_infecciosa"] = np.where(d["infecciosas"] >= 0, "Infecciosa alta", "Infecciosa baja")
    d["carga_nutricional"] = np.where(d["nutricionales"] >= 0, "Nutricional alta", "Nutricional baja")
    cuad = (d.groupby(["nivel_pobreza", "carga_infecciosa", "carga_nutricional"])
            .agg(tasa_desercion_media=("tasa_desercion", "mean"), observaciones=("tasa_desercion", "size")).reset_index())
    guardar_tabla(a_spark(cuad), SCHEMA_GOLD, "analisis_cuadrantes")
    guardar_tabla(a_spark(d[["cod_dpto", "anio", "perfil_epidemiologico", "nivel_pobreza", "carga_infecciosa", "carga_nutricional"]]),
                  SCHEMA_GOLD, "clasificacion_territorial")
else:
    verificar("oro", "analisis_interaccion", "indicador de pobreza disponible", False, dimension="completitud")

Indicador de pobreza para la interacción: pob_indice_de_pobreza_multidimensional_ipm
  ✔ desercion_escolar.capa_oro.analisis_interaccion (overwrite) → 12 filas


termino,coeficiente,error_estandar_cluster,t,p_valor,ic95_inf,ic95_sup,especificacion,r2,n,variable_pobreza
intercepto,3.321826942858954,0.44723023031866,7.427554574054821,1.8828420014600366E-8,2.4108487745195353,4.232805111198373,A: efectos fijos de año,0.25559079631805826,231,pob_indice_de_pobreza_multidimensional_ipm
infecciosas,0.5199760738533074,0.36329518596309807,1.4312770825048153,0.16204543461309315,-0.22003200395346523,1.2599841516600798,A: efectos fijos de año,0.25559079631805826,231,pob_indice_de_pobreza_multidimensional_ipm
nutricionales,0.5056164859895925,0.5396940530379447,0.9368576198745768,0.35584905294239955,-0.5937043259105214,1.6049372978897063,A: efectos fijos de año,0.25559079631805826,231,pob_indice_de_pobreza_multidimensional_ipm
pobreza,0.16434742615085177,0.24410224587725146,0.6732728966102796,0.5056075390831438,-0.33287257769001777,0.6615674299917212,A: efectos fijos de año,0.25559079631805826,231,pob_indice_de_pobreza_multidimensional_ipm
infecciosas_x_pobreza,0.14527884584982992,0.14616631952159023,0.993928329900725,0.3277145110724865,-0.1524522040745403,0.44300989577420014,A: efectos fijos de año,0.25559079631805826,231,pob_indice_de_pobreza_multidimensional_ipm
nutricionales_x_pobreza,0.11892446111336849,0.14037620669072082,0.8471838918926256,0.4031844788842216,-0.16701251492340766,0.40486143715014467,A: efectos fijos de año,0.25559079631805826,231,pob_indice_de_pobreza_multidimensional_ipm
intercepto,3.6888238661321253,0.23337088017873603,15.80670160435997,1.1287202438692588E-16,3.213462938903426,4.164184793360825,B: efectos fijos de departamento y año,0.7671448855831418,231,pob_indice_de_pobreza_multidimensional_ipm
infecciosas,0.1531423497669656,0.24759839382639784,0.6185110791725914,0.540616503885105,-0.3511990744051898,0.657483773939121,B: efectos fijos de departamento y año,0.7671448855831418,231,pob_indice_de_pobreza_multidimensional_ipm
nutricionales,-0.005249188141686956,0.21252404335402186,-0.02469926723981472,0.9804482236103378,-0.4381464983364542,0.4276481220530803,B: efectos fijos de departamento y año,0.7671448855831418,231,pob_indice_de_pobreza_multidimensional_ipm
pobreza,-0.24609431239455676,0.2824053927189679,-0.871422142563171,0.3900148649661356,-0.8213352731967669,0.32914664840765345,B: efectos fijos de departamento y año,0.7671448855831418,231,pob_indice_de_pobreza_multidimensional_ipm


  ✔ desercion_escolar.capa_oro.analisis_cuadrantes (overwrite) → 7 filas
  ✔ desercion_escolar.capa_oro.clasificacion_territorial (overwrite) → 231 filas


## 8. Tablas de hechos para Power BI (modelo estrella)

In [ ]:
# Educación 2011-2024 + promedio nacional ponderado por población 5-16
w_anio = Window.partitionBy("anio")
fact_edu = (edu.join(dim_dpto.select("cod_dpto", "region"), "cod_dpto")
            .withColumn("tasa_desercion_nacional_ponderada",
                        F.sum(F.col("tasa_desercion") * F.col("poblacion_5_16")).over(w_anio) / F.sum("poblacion_5_16").over(w_anio))
            .withColumn("brecha_vs_nacional", F.col("tasa_desercion") - F.col("tasa_desercion_nacional_ponderada")))
guardar_tabla(fact_edu, SCHEMA_GOLD, "fact_educacion", particion="anio")

fact_siv = (leer_tabla(SCHEMA_SILVER, "sivigila_departamento_anio_perfil")
            .join(pob_denom, ["cod_dpto", "anio"], "left")
            .withColumn("tasa_por_1000_nna", F.col("casos") / F.col("poblacion_5_16") * 1000)
            .drop("poblacion_5_16"))
guardar_tabla(fact_siv, SCHEMA_GOLD, "fact_sivigila_perfil", particion="anio")

guardar_tabla(pob.join(cat_pob.select("indicador_id", F.col("indicador_id").isin(INDICADORES_SEL).alias("seleccionado")), "indicador_id"),
              SCHEMA_GOLD, "fact_pobreza", particion="anio")

persistir_calidad()
print("Capa Oro (datos y análisis) completada ✔")

  ✔ desercion_escolar.capa_oro.fact_educacion (overwrite) → 462 filas
  ✔ desercion_escolar.capa_oro.fact_sivigila_perfil (overwrite) → 2,673 filas
  ✔ desercion_escolar.capa_oro.fact_pobreza (overwrite) → 19,776 filas
  ✔ desercion_escolar.gobierno_datos.control_calidad (append) → 186 filas
Capa Oro (datos y análisis) completada ✔


## 04_Modelado_Evaluacion

<!-- merged from: 04_Modelado_Evaluacion.ipynb (Jupy Tools) -->

# 04 · Modelado y evaluación — backtesting temporal, selección por validación y prueba 2024 única

**CRISP-DM:** Modelado + Evaluación · **Medallion:** Oro (consumo analítico)

### Protocolo (corrige los hallazgos de la sección 5 del informe)
| Paso | Implementación |
|---|---|
| Líneas base | **Persistencia** (tasa de *t−1*) y **media móvil de 3 años**. Un modelo solo se acepta si las supera. |
| Validación | *Backtesting* con ventana expansiva: entrena 2018…*v−1*, valida en *v* ∈ {2020, 2021, 2022, 2023}. |
| Selección | Familia e hiperparámetros por **RMSE medio de validación**. El año 2024 **no interviene** en la selección. |
| Prueba | El ganador se reentrena con 2018–2023 y se evalúa **una sola vez** en 2024, con IC 95 % por *bootstrap* de departamentos. |
| Sin fuga | Imputación (mediana) y estandarización viven dentro del `Pipeline` y se ajustan **solo con el entrenamiento de cada pliegue**. |
| Incertidumbre | Intervalos de predicción del 90 % a partir de los residuos fuera de muestra del *backtesting* (conformal empírico). |
| Interpretabilidad | Importancia por **permutación** fuera de muestra (agnóstica al modelo) + importancia nativa / coeficientes estandarizados. |
| Equidad territorial | Métricas por departamento, región y nivel de pobreza. |
| Registro | `modelo_registro` (versión, hiperparámetros, variables, años, métricas, ruta, versión de datos) + MLflow si está disponible. |

Familias: **ElasticNet** (regresión lineal regularizada, línea base paramétrica), **Random Forest** y **Gradient Boosted Trees**
(equivalente en Spark ML a XGBoost/LightGBM). La variable objetivo es continua (tasa), por lo que se usa regresión;
la regresión logística mencionada en el anteproyecto no aplica a un objetivo continuo agregado.

In [ ]:
%run ./00_Config_Utilidades

Entorno Databricks: True | formato: delta | ejecución: 20261005T162942-18e758


Utilidades cargadas ✔


In [ ]:
import numpy as np
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor

MODO_RAPIDO = _widget("modo_rapido", "no").lower() in ("si", "sí", "true", "1", "yes")
rng = np.random.default_rng(SEMILLA)
MODELO_VERSION = f"v{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"
print(f"Versión de modelo: {MODELO_VERSION} | modo rápido: {MODO_RAPIDO}")

fs = leer_tabla(SCHEMA_GOLD, "feature_store_desercion")
version_datos = fs.select("_version_datos").first()[0]
dim_dpto_pd = df_departamentos().toPandas()

Versión de modelo: v20261005162948 | modo rápido: False


## 1. Conjuntos de variables por escenario

In [ ]:
todas = fs.columns
EDU = [c for c in todas if c.startswith("edu_")]
SIV_T = [c for c in todas if c.startswith("siv_") and not c.endswith("_lag1")]
SIV_LAG = [c for c in todas if c.startswith("siv_") and c.endswith("_lag1")]

PATRONES_POB = [r"indice_de_pobreza_multidimensional", r"pobreza_monetaria", r"pobreza_extrema|indigencia", r"gini",
                r"bajo_logro_educativo", r"analfabetismo", r"inasistencia_escolar", r"rezago_escolar",
                r"trabajo_infantil", r"trabajo_informal", r"desempleo", r"aseguramiento", r"barreras",
                r"fuente_de_agua", r"excretas|alcantarillado", r"paredes", r"pisos", r"hacinamiento",
                r"necesidades_basicas|\bnbi\b", r"primera_infancia"]
pob_cols = [c for c in todas if c.startswith("pob_") and c != "pob_antiguedad_media_anios"]
POB = []
for pat in PATRONES_POB:
    m = [c for c in pob_cols if re.search(pat, c)]
    nuevos = [c for c in m if "antigua" not in c] or m
    POB += [c for c in nuevos if c not in POB]
if not POB:
    POB = pob_cols
pdf_fs = fs.toPandas()      # ≤ 264 filas: permite calcular coberturas por pliegue sin costo
if len(POB) > 25:
    POB = sorted(POB, key=lambda c: pdf_fs[c].isna().mean())[:25]

ESCENARIOS = {
    "nowcast": {"features": EDU + SIV_T + POB, "anio_min_train": ANIO_MIN,
                "descripcion": "Tasa del año t con SIVIGILA de t e información publicada hasta t−1"},
    "pronostico": {"features": EDU + SIV_LAG + POB, "anio_min_train": ANIO_MIN + 1,
                   "descripcion": "Tasa del año t con información disponible al cierre de t−1 (produce 2025)"},
}
# Nota metodológica: la bandera `periodo_pandemia` NO se usa como predictor. 2020 (cierre de la presencialidad, deserción
# registrada baja) y 2021 (retorno, repunte) tienen efectos de signo opuesto; una sola bandera aprende el efecto de 2020
# y lo aplica a 2021, y una bandera por año no es estimable fuera de muestra (es constante en el entrenamiento). El
# choque pandémico se documenta en el análisis descriptivo y en las métricas por pliegue.
for k, v in ESCENARIOS.items():
    print(f"{k}: {len(v['features'])} variables candidatas")

model_df = fs.select("cod_dpto", "departamento", "region", "anio", F.col("tasa_desercion").cast("double"),
                     *[F.col(c).cast("double").alias(c) for c in sorted(set(EDU + SIV_T + SIV_LAG + POB + ["periodo_pandemia"]))])
model_df = cachear(model_df.withColumn("_base_lag1", F.col("edu_tasa_desercion_lag1"))
                   .withColumn("delta_desercion", F.col("tasa_desercion") - F.col("edu_tasa_desercion_lag1")))
model_df.count()

nowcast: 63 variables candidatas
pronostico: 63 variables candidatas


264

## 2. Familias de modelos, rejillas y utilidades de evaluación

In [ ]:
# Cada configuración define además el "objetivo de entrenamiento":
#   nivel → se modela la tasa directamente;
#   delta → se modela el cambio respecto al año anterior (tasa_t − tasa_{t−1}) y la predicción es tasa_{t−1} + Δ̂.
# El modelo "delta" se encoge hacia la línea base de persistencia y solo se aparta de ella si las variables aportan
# señal; con 33 departamentos y pocos años es la formulación más robusta frente al sobreajuste. La validación decide.
def _lr(p, label):  return LinearRegression(featuresCol="features", labelCol=label, maxIter=300, standardization=False, **p)
def _rf(p, label):  return RandomForestRegressor(featuresCol="features", labelCol=label, seed=SEMILLA, featureSubsetStrategy="onethird", **p)
def _gbt(p, label): return GBTRegressor(featuresCol="features", labelCol=label, seed=SEMILLA, subsamplingRate=0.8, **p)

if MODO_RAPIDO:
    FAMILIAS = {
        "ElasticNet": (_lr, [{"objetivo": "delta", "regParam": 0.1, "elasticNetParam": 0.5}]),
        "RandomForest": (_rf, [{"objetivo": "delta", "numTrees": 50, "maxDepth": 4, "minInstancesPerNode": 3}]),
        "GBT": (_gbt, [{"objetivo": "nivel", "maxIter": 30, "maxDepth": 2, "stepSize": 0.1}]),
    }
else:
    FAMILIAS = {
        "ElasticNet": (_lr, [{"objetivo": o, "regParam": r, "elasticNetParam": a}
                             for o in ["nivel", "delta"] for r in [0.01, 0.1, 0.5, 1.0] for a in [0.0, 0.5, 1.0]]),
        "RandomForest": (_rf, [{"objetivo": o, "numTrees": 300, "maxDepth": d, "minInstancesPerNode": m}
                               for o in ["nivel", "delta"] for d in [3, 5] for m in [3, 6]]),
        "GBT": (_gbt, [{"objetivo": o, "maxIter": it, "maxDepth": d, "stepSize": 0.05}
                       for o in ["nivel", "delta"] for it in [50, 150] for d in [2, 3]]),
    }


def metricas(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = ~(np.isnan(y) | np.isnan(yhat))
    y, yhat = y[m], yhat[m]
    if len(y) == 0:
        return {"n": 0, "mae": np.nan, "rmse": np.nan, "r2": np.nan, "sesgo": np.nan}
    e = yhat - y
    sst = ((y - y.mean()) ** 2).sum()
    return {"n": int(len(y)), "mae": float(np.abs(e).mean()), "rmse": float(np.sqrt((e ** 2).mean())),
            "r2": float(1 - (e ** 2).sum() / sst) if sst > 0 else np.nan, "sesgo": float(e.mean())}


def features_utiles(feats, anios_train):
    """Descarta variables con > 40 % de nulos o sin variación en el entrenamiento del pliegue."""
    tr = pdf_fs[pdf_fs.anio.isin(anios_train) & pdf_fs.tasa_desercion.notna()]
    return [f for f in feats if tr[f].notna().mean() >= 0.6 and tr[f].nunique(dropna=True) > 1]


class ModeloDesercion:
    """Envoltorio de un PipelineModel que reconstruye la tasa cuando el objetivo de entrenamiento es el cambio anual."""
    def __init__(self, modelo, objetivo):
        self.modelo, self.objetivo = modelo, objetivo
        self.stages = modelo.stages

    def transform(self, df):
        out = self.modelo.transform(df)
        if self.objetivo == "delta":
            out = out.withColumn("prediction", F.col("prediction") + F.col("_base_lag1"))
        return out

    def write(self):
        return self.modelo.write()


def ajustar(familia, params, feats, anios_train):
    params = dict(params)
    objetivo = params.pop("objetivo", "nivel")
    label = "delta_desercion" if objetivo == "delta" else "tasa_desercion"
    imp = [f"{f}__imp" for f in feats]
    pipe = Pipeline(stages=[
        Imputer(inputCols=feats, outputCols=imp, strategy="median"),
        VectorAssembler(inputCols=imp, outputCol="features_raw", handleInvalid="keep"),
        StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True),
        FAMILIAS[familia][0](params, label)])
    train = model_df.filter(F.col("anio").isin(list(anios_train)) & F.col(label).isNotNull())
    return ModeloDesercion(pipe.fit(train), objetivo)


def predecir(modelo, anios):
    return (modelo.transform(model_df.filter(F.col("anio").isin(list(anios))))
            .select("cod_dpto", "anio", "tasa_desercion", F.col("prediction").alias("prediccion")).toPandas())


def baselines(anios):
    d = pdf_fs[pdf_fs.anio.isin(anios) & pdf_fs.tasa_desercion.notna()][["cod_dpto", "anio", "tasa_desercion",
                                                                          "edu_tasa_desercion_lag1", "edu_tasa_desercion_media3"]].copy()
    d["Persistencia (t−1)"] = d["edu_tasa_desercion_lag1"]
    d["Media móvil 3 años"] = d["edu_tasa_desercion_media3"].fillna(d["edu_tasa_desercion_lag1"])
    return d

## 3. *Backtesting*, selección, prueba única y modelo final por escenario

In [ ]:
tablas = {k: [] for k in ["grid", "metricas", "predicciones", "importancia", "registro", "depto", "subgrupo"]}


def bootstrap_rmse(df, col_pred, n=1000):
    """IC 95 % del RMSE remuestreando departamentos (unidad de independencia)."""
    deptos = df.cod_dpto.unique()
    vals = []
    for _ in range(n):
        s = rng.choice(deptos, size=len(deptos), replace=True)
        d = pd.concat([df[df.cod_dpto == x] for x in s])
        vals.append(np.sqrt(((d[col_pred] - d.tasa_desercion) ** 2).mean()))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


for escenario, cfg in ESCENARIOS.items():
    print(f"\n{'=' * 70}\nESCENARIO: {escenario} — {cfg['descripcion']}\n{'=' * 70}")
    a0 = cfg["anio_min_train"]
    folds = [v for v in ANIOS_VALIDACION if v - a0 >= 2]
    feats_base = cfg["features"]

    # ---------- 3.1 Backtesting de toda la rejilla ----------
    oos = {}
    for familia, (_, rejilla) in FAMILIAS.items():
        for i, params in enumerate(rejilla):
            partes = []
            for v in folds:
                tr_anios = list(range(a0, v))
                feats = features_utiles(feats_base, tr_anios)
                p = predecir(ajustar(familia, params, feats, tr_anios), [v])
                partes.append(p)
            dfp = pd.concat(partes)
            por_fold = [metricas(g.tasa_desercion, g.prediccion)["rmse"] for _, g in dfp.groupby("anio")]
            m = metricas(dfp.tasa_desercion, dfp.prediccion)
            oos[(familia, i)] = dfp
            tablas["grid"].append([escenario, familia, i, json.dumps(params), float(np.mean(por_fold)), float(np.std(por_fold)), m["mae"], m["r2"]])
            print(f"  {familia:13s} cfg{i:<2d} RMSE val medio={np.mean(por_fold):.4f} ± {np.std(por_fold):.4f}  {params}")

    grid_esc = pd.DataFrame([g for g in tablas["grid"] if g[0] == escenario],
                            columns=["escenario", "familia", "cfg", "params", "rmse_val_medio", "rmse_val_std", "mae_val", "r2_val"])
    mejores = grid_esc.loc[grid_esc.groupby("familia")["rmse_val_medio"].idxmin()].sort_values("rmse_val_medio")
    ganador = mejores.iloc[0]
    fam_g, cfg_g, params_g = ganador.familia, int(ganador.cfg), json.loads(ganador.params)
    print(f"\n  ➜ Ganador por validación: {fam_g} cfg{cfg_g} {params_g}")

    # Métricas de validación por pliegue (mejor de cada familia + líneas base)
    bl_val = baselines(folds)
    for _, r in mejores.iterrows():
        dfp = oos[(r.familia, int(r.cfg))]
        for v, g in dfp.groupby("anio"):
            tablas["metricas"].append([escenario, r.familia, "validacion", int(v), *metricas(g.tasa_desercion, g.prediccion).values()])
        tablas["metricas"].append([escenario, r.familia, "validacion_global", None, *metricas(dfp.tasa_desercion, dfp.prediccion).values()])
    for b in ["Persistencia (t−1)", "Media móvil 3 años"]:
        for v, g in bl_val.groupby("anio"):
            tablas["metricas"].append([escenario, b, "validacion", int(v), *metricas(g.tasa_desercion, g[b]).values()])
        tablas["metricas"].append([escenario, b, "validacion_global", None, *metricas(bl_val.tasa_desercion, bl_val[b]).values()])

    rmse_val_g = metricas(oos[(fam_g, cfg_g)].tasa_desercion, oos[(fam_g, cfg_g)].prediccion)["rmse"]
    rmse_val_p = metricas(bl_val.tasa_desercion, bl_val["Persistencia (t−1)"])["rmse"]
    verificar("modelado", escenario, "el ganador supera a la persistencia en validación", rmse_val_g < rmse_val_p,
              valor=round(rmse_val_g, 4), umbral=f"< {rmse_val_p:.4f}", dimension="desempeño",
              detalle="si falla, la persistencia es el modelo recomendado y así se reporta")

    # ---------- 3.2 Prueba única 2024 (solo en nowcast; el pronóstico se evalúa igual para comparabilidad) ----------
    tr_test = list(range(a0, ANIO_TEST))
    feats_test = features_utiles(feats_base, tr_test)
    pred_test = None
    for _, r in mejores.iterrows():
        modelo_t = ajustar(r.familia, json.loads(r.params), feats_test, tr_test)
        p = predecir(modelo_t, [ANIO_TEST])
        tablas["metricas"].append([escenario, r.familia, "prueba", ANIO_TEST, *metricas(p.tasa_desercion, p.prediccion).values()])
        if r.familia == fam_g:
            pred_test, modelo_test = p, modelo_t
    bl_test = baselines([ANIO_TEST])
    for b in ["Persistencia (t−1)", "Media móvil 3 años"]:
        tablas["metricas"].append([escenario, b, "prueba", ANIO_TEST, *metricas(bl_test.tasa_desercion, bl_test[b]).values()])

    mt = metricas(pred_test.tasa_desercion, pred_test.prediccion)
    comp = pred_test.merge(bl_test[["cod_dpto", "Persistencia (t−1)"]], on="cod_dpto")
    ic_mod = bootstrap_rmse(comp, "prediccion", 300 if MODO_RAPIDO else 1000)
    ic_per = bootstrap_rmse(comp, "Persistencia (t−1)", 300 if MODO_RAPIDO else 1000)
    rmse_per_test = metricas(comp.tasa_desercion, comp["Persistencia (t−1)"])["rmse"]
    mejora = 100 * (rmse_per_test - mt["rmse"]) / rmse_per_test if rmse_per_test else np.nan
    print(f"  Prueba {ANIO_TEST}: RMSE={mt['rmse']:.4f} IC95=[{ic_mod[0]:.4f}, {ic_mod[1]:.4f}] | "
          f"persistencia={rmse_per_test:.4f} IC95=[{ic_per[0]:.4f}, {ic_per[1]:.4f}] | mejora={mejora:.1f} %")

    # ---------- 3.3 Intervalos de predicción (conformal empírico con residuos fuera de muestra) ----------
    oos_g = pd.concat([oos[(fam_g, cfg_g)].assign(tipo_prediccion="backtest_validacion"),
                       pred_test.assign(tipo_prediccion=f"prueba_{ANIO_TEST}")])
    q90 = float(np.quantile(np.abs(oos[(fam_g, cfg_g)].prediccion - oos[(fam_g, cfg_g)].tasa_desercion), 0.90))
    print(f"  Semiamplitud del intervalo 90 %: ±{q90:.3f} pp")

    # ---------- 3.4 Métricas por departamento, región y pobreza (fuera de muestra) ----------
    oos_g = oos_g.merge(dim_dpto_pd[["cod_dpto", "departamento", "region"]], on="cod_dpto", how="left")
    for (cod, dep), g in oos_g.groupby(["cod_dpto", "departamento"]):
        tablas["depto"].append([escenario, int(cod), dep, *metricas(g.tasa_desercion, g.prediccion).values()])
    for reg, g in oos_g.groupby("region"):
        tablas["subgrupo"].append([escenario, "Región", reg, *metricas(g.tasa_desercion, g.prediccion).values()])
    var_pob = next((c for c in POB if "multidimensional" in c and "cabecera" not in c and "rural" not in c), POB[0] if POB else None)
    if var_pob:
        med = pdf_fs.loc[pdf_fs.anio < ANIO_TEST, var_pob].median()
        niv = pdf_fs[["cod_dpto", "anio", var_pob]].assign(nivel=lambda x: np.where(x[var_pob] >= med, "Pobreza alta", "Pobreza baja"))
        tmp = oos_g.merge(niv[["cod_dpto", "anio", "nivel"]], on=["cod_dpto", "anio"], how="left")
        for nv, g in tmp.groupby("nivel"):
            tablas["subgrupo"].append([escenario, f"Pobreza ({var_pob})", nv, *metricas(g.tasa_desercion, g.prediccion).values()])

    # ---------- 3.5 Importancia por permutación fuera de muestra (una transformación por pliegue) ----------
    repeticiones = 1 if MODO_RAPIDO else 5
    acumulado = {}
    for v in folds:
        tr_anios = list(range(a0, v))
        feats = features_utiles(feats_base, tr_anios)
        modelo_v = ajustar(fam_g, params_g, feats, tr_anios)
        val = model_df.filter(F.col("anio") == v).toPandas()
        base_v = predecir(modelo_v, [v])
        rmse0 = metricas(base_v.tasa_desercion, base_v.prediccion)["rmse"]
        bloques = []
        for f in feats:
            for rpt in range(repeticiones):
                b = val.copy()
                b[f] = rng.permutation(b[f].values)
                b["_feat"], b["_rep"] = f, rpt
                bloques.append(b)
        perm = pd.concat(bloques, ignore_index=True)
        cols_num = [c for c in model_df.columns if c not in ("departamento", "region")]
        esquema_perm = T.StructType([model_df.schema[c] for c in cols_num]
                                    + [T.StructField("_feat", T.StringType()), T.StructField("_rep", T.IntegerType())])
        perm_sdf = a_spark(perm[cols_num + ["_feat", "_rep"]], esquema_perm)
        res = modelo_v.transform(perm_sdf).select("_feat", "_rep", "tasa_desercion", "prediction").toPandas()
        for (f, rpt), g in res.groupby(["_feat", "_rep"]):
            acumulado.setdefault(f, []).append(metricas(g.tasa_desercion, g.prediction)["rmse"] - rmse0)

    # ---------- 3.6 Modelo final (todo el histórico) ----------
    tr_full = list(range(a0, ANIO_MAX + 1))
    feats_final = features_utiles(feats_base, tr_full)
    modelo_final = ajustar(fam_g, params_g, feats_final, tr_full)
    estimador = modelo_final.stages[-1]
    if hasattr(estimador, "featureImportances"):
        nativa = dict(zip(feats_final, estimador.featureImportances.toArray()))
    else:
        nativa = dict(zip(feats_final, estimador.coefficients.toArray()))
    for f in sorted(set(acumulado) | set(nativa)):
        vals = acumulado.get(f, [])
        tablas["importancia"].append([escenario, fam_g, f, float(np.mean(vals)) if vals else None,
                                      float(np.std(vals)) if vals else None, float(nativa[f]) if f in nativa else None,
                                      "coeficiente estandarizado" if not hasattr(estimador, "featureImportances") else "featureImportances"])

    ruta_modelo = f"{MODELOS_PATH}/{escenario}/{MODELO_VERSION}"  # objetivo de entrenamiento en hiperparametros
    try:
        modelo_final.write().overwrite().save(ruta_modelo)
    except Exception as e:
        print(f"  (no se pudo serializar el modelo en {ruta_modelo}: {e})")
        ruta_modelo = None

    # ---------- 3.7 Predicciones para el tablero ----------
    if escenario == "nowcast":
        ajuste = predecir(modelo_final, list(range(ANIO_MIN, ANIO_MAX + 1))).assign(tipo_prediccion="ajuste_historico")
        preds = pd.concat([oos_g[["cod_dpto", "anio", "tasa_desercion", "prediccion", "tipo_prediccion"]], ajuste])
    else:
        futuro = predecir(modelo_final, [ANIO_PRONOSTICO]).assign(tipo_prediccion=f"pronostico_{ANIO_PRONOSTICO}")
        preds = pd.concat([oos_g[["cod_dpto", "anio", "tasa_desercion", "prediccion", "tipo_prediccion"]], futuro])
    preds = preds.merge(pdf_fs[["cod_dpto", "anio", "edu_tasa_desercion_lag1"]], on=["cod_dpto", "anio"], how="left")
    preds["escenario"], preds["modelo"], preds["modelo_version"] = escenario, fam_g, MODELO_VERSION
    preds["limite_inferior_90"] = (preds.prediccion - q90).clip(lower=0)
    preds["limite_superior_90"] = preds.prediccion + q90
    preds["es_fuera_de_muestra"] = preds.tipo_prediccion != "ajuste_historico"
    tablas["predicciones"].append(preds)

    # ---------- 3.8 Registro del modelo ----------
    tablas["registro"].append([MODELO_VERSION, escenario, cfg["descripcion"], fam_g, json.dumps(params_g), json.dumps(feats_final),
                               len(feats_final), f"{a0}-{ANIO_MAX}", json.dumps(folds), float(ganador.rmse_val_medio),
                               float(rmse_val_p), mt["mae"], mt["rmse"], mt["r2"], ic_mod[0], ic_mod[1], float(rmse_per_test),
                               float(mejora), bool(rmse_val_g < rmse_val_p and mt["rmse"] < rmse_per_test), q90,
                               ruta_modelo, version_datos, EJECUCION_ID, datetime.now()])

    try:
        import mlflow
        if EN_DATABRICKS:
            mlflow.set_experiment("/Shared/desercion_escolar_departamental")
        with mlflow.start_run(run_name=f"{escenario}-{fam_g}-{MODELO_VERSION}"):
            mlflow.log_params({"escenario": escenario, "familia": fam_g, **{f"hp_{k}": v for k, v in params_g.items()},
                               "n_features": len(feats_final), "version_datos": version_datos[:250]})
            mlflow.log_metrics({"rmse_val_medio": float(ganador.rmse_val_medio), "rmse_test": mt["rmse"],
                                "mae_test": mt["mae"], "r2_test": mt["r2"], "rmse_persistencia_test": float(rmse_per_test)})
    except Exception as e:
        print(f"  (MLflow no disponible: {e})")


ESCENARIO: nowcast — Tasa del año t con SIVIGILA de t e información publicada hasta t−1
  ElasticNet    cfg0  RMSE val medio=3.0447 ± 1.9781  {'objetivo': 'nivel', 'regParam': 0.01, 'elasticNetParam': 0.0}
  ElasticNet    cfg1  RMSE val medio=2.6700 ± 1.3465  {'objetivo': 'nivel', 'regParam': 0.01, 'elasticNetParam': 0.5}
  ElasticNet    cfg2  RMSE val medio=2.1906 ± 0.6323  {'objetivo': 'nivel', 'regParam': 0.01, 'elasticNetParam': 1.0}
  ElasticNet    cfg3  RMSE val medio=2.2982 ± 0.8376  {'objetivo': 'nivel', 'regParam': 0.1, 'elasticNetParam': 0.0}
  ElasticNet    cfg4  RMSE val medio=1.6324 ± 0.1871  {'objetivo': 'nivel', 'regParam': 0.1, 'elasticNetParam': 0.5}
  ElasticNet    cfg5  RMSE val medio=1.5804 ± 0.2901  {'objetivo': 'nivel', 'regParam': 0.1, 'elasticNetParam': 1.0}
  ElasticNet    cfg6  RMSE val medio=1.7187 ± 0.1263  {'objetivo': 'nivel', 'regParam': 0.5, 'elasticNetParam': 0.0}
  ElasticNet    cfg7  RMSE val medio=1.5074 ± 0.4091  {'objetivo': 'nivel', 'regParam': 0

2026/10/05 17:49:47 INFO mlflow.tracking.fluent: Experiment with name '/Shared/desercion_escolar_departamental' does not exist. Creating a new experiment.



ESCENARIO: pronostico — Tasa del año t con información disponible al cierre de t−1 (produce 2025)
  ElasticNet    cfg0  RMSE val medio=3.6072 ± 2.0759  {'objetivo': 'nivel', 'regParam': 0.01, 'elasticNetParam': 0.0}
  ElasticNet    cfg1  RMSE val medio=2.7445 ± 0.9784  {'objetivo': 'nivel', 'regParam': 0.01, 'elasticNetParam': 0.5}
  ElasticNet    cfg2  RMSE val medio=2.3509 ± 0.6741  {'objetivo': 'nivel', 'regParam': 0.01, 'elasticNetParam': 1.0}
  ElasticNet    cfg3  RMSE val medio=2.0683 ± 0.6344  {'objetivo': 'nivel', 'regParam': 0.1, 'elasticNetParam': 0.0}
  ElasticNet    cfg4  RMSE val medio=1.7662 ± 0.6498  {'objetivo': 'nivel', 'regParam': 0.1, 'elasticNetParam': 0.5}
  ElasticNet    cfg5  RMSE val medio=1.6209 ± 0.5765  {'objetivo': 'nivel', 'regParam': 0.1, 'elasticNetParam': 1.0}
  ElasticNet    cfg6  RMSE val medio=1.5647 ± 0.5186  {'objetivo': 'nivel', 'regParam': 0.5, 'elasticNetParam': 0.0}
  ElasticNet    cfg7  RMSE val medio=1.4897 ± 0.5691  {'objetivo': 'nivel', 're

## 4. Persistencia de resultados en Oro

In [ ]:
M_COLS = "escenario STRING, modelo STRING, conjunto STRING, anio INT, n INT, mae DOUBLE, rmse DOUBLE, r2 DOUBLE, sesgo DOUBLE"
guardar_tabla(a_spark(pd.DataFrame(tablas["grid"], columns=["escenario", "familia", "cfg", "hiperparametros", "rmse_val_medio", "rmse_val_std", "mae_val", "r2_val"]),
                      "escenario STRING, familia STRING, cfg INT, hiperparametros STRING, rmse_val_medio DOUBLE, rmse_val_std DOUBLE, mae_val DOUBLE, r2_val DOUBLE"),
              SCHEMA_GOLD, "modelo_resultados_grid")
guardar_tabla(a_spark(pd.DataFrame(tablas["metricas"]), M_COLS), SCHEMA_GOLD, "modelo_metricas")
guardar_tabla(a_spark(pd.DataFrame(tablas["depto"]), "escenario STRING, cod_dpto INT, departamento STRING, n INT, mae DOUBLE, rmse DOUBLE, r2 DOUBLE, sesgo DOUBLE"),
              SCHEMA_GOLD, "modelo_metricas_departamento")
guardar_tabla(a_spark(pd.DataFrame(tablas["subgrupo"]), "escenario STRING, tipo_subgrupo STRING, subgrupo STRING, n INT, mae DOUBLE, rmse DOUBLE, r2 DOUBLE, sesgo DOUBLE"),
              SCHEMA_GOLD, "modelo_metricas_subgrupo")

imp = pd.DataFrame(tablas["importancia"], columns=["escenario", "modelo", "variable", "importancia_permutacion", "importancia_permutacion_std", "importancia_nativa", "tipo_importancia_nativa"])
imp["ranking"] = imp.groupby("escenario")["importancia_permutacion"].rank(ascending=False, method="first")
dicc = leer_tabla(SCHEMA_GOLD, "diccionario_variables").toPandas()
imp = imp.merge(dicc[["variable", "fuente", "grupo"]], on="variable", how="left")
guardar_tabla(a_spark(imp, "escenario STRING, modelo STRING, variable STRING, importancia_permutacion DOUBLE, importancia_permutacion_std DOUBLE, importancia_nativa DOUBLE, tipo_importancia_nativa STRING, ranking DOUBLE, fuente STRING, grupo STRING"),
              SCHEMA_GOLD, "modelo_importancia")

ESQ_REG = ("modelo_version STRING, escenario STRING, descripcion STRING, familia STRING, hiperparametros STRING, variables STRING, "
           "n_variables INT, anios_entrenamiento STRING, anios_validacion STRING, rmse_validacion DOUBLE, rmse_validacion_persistencia DOUBLE, "
           "mae_prueba DOUBLE, rmse_prueba DOUBLE, r2_prueba DOUBLE, rmse_prueba_ic95_inf DOUBLE, rmse_prueba_ic95_sup DOUBLE, "
           "rmse_prueba_persistencia DOUBLE, mejora_vs_persistencia_pct DOUBLE, supera_linea_base BOOLEAN, semiamplitud_intervalo_90 DOUBLE, "
           "ruta_modelo STRING, version_datos STRING, ejecucion_id STRING, fecha_entrenamiento TIMESTAMP")
guardar_tabla(spark.createDataFrame(tablas["registro"], ESQ_REG), SCHEMA_GOLD, "modelo_registro", modo="append")

  ✔ desercion_escolar.capa_oro.modelo_resultados_grid (overwrite) → 80 filas
  ✔ desercion_escolar.capa_oro.modelo_metricas (overwrite) → 55 filas
  ✔ desercion_escolar.capa_oro.modelo_metricas_departamento (overwrite) → 66 filas
  ✔ desercion_escolar.capa_oro.modelo_metricas_subgrupo (overwrite) → 16 filas
  ✔ desercion_escolar.capa_oro.modelo_importancia (overwrite) → 120 filas
  ✔ desercion_escolar.capa_oro.modelo_registro (append) → 2 filas


2

### Tabla de predicciones para Power BI (con prioridad territorial)
Umbrales de prioridad = terciles de la tasa observada 2018–2023 (no usan el periodo evaluado).
`tendencia_esperada` compara el intervalo de predicción con la tasa del año anterior.

In [ ]:
preds = pd.concat(tablas["predicciones"], ignore_index=True)
hist = pdf_fs[(pdf_fs.anio < ANIO_TEST) & pdf_fs.tasa_desercion.notna()].tasa_desercion
p33, p66 = float(hist.quantile(0.33)), float(hist.quantile(0.66))
preds["nivel_prioridad"] = np.select([preds.prediccion >= p66, preds.prediccion >= p33], ["CRITICA", "ALTA"], "NORMAL")
preds["orden_prioridad"] = preds.nivel_prioridad.map({"CRITICA": 1, "ALTA": 2, "NORMAL": 3})
preds["tendencia_esperada"] = np.select(
    [preds.limite_inferior_90 > preds.edu_tasa_desercion_lag1, preds.limite_superior_90 < preds.edu_tasa_desercion_lag1],
    ["Aumento", "Disminución"], "Estable")
preds["error"] = preds.prediccion - preds.tasa_desercion
preds["error_absoluto"] = preds.error.abs()
preds["error_porcentual"] = np.where(preds.tasa_desercion > 0, 100 * preds.error_absoluto / preds.tasa_desercion, np.nan)
preds["dentro_intervalo_90"] = np.where(preds.tasa_desercion.notna(),
                                        (preds.tasa_desercion >= preds.limite_inferior_90) & (preds.tasa_desercion <= preds.limite_superior_90), None)
preds = preds.rename(columns={"tasa_desercion": "tasa_real", "edu_tasa_desercion_lag1": "tasa_anio_anterior"})
preds = preds.merge(dim_dpto_pd[["cod_dpto", "departamento", "region"]], on="cod_dpto", how="left")
preds["umbral_alta"], preds["umbral_critica"] = p33, p66

cobertura = preds[preds.es_fuera_de_muestra & preds.tasa_real.notna()].groupby("escenario")["dentro_intervalo_90"].mean()
for esc, c in cobertura.items():
    verificar("modelado", esc, "cobertura empírica del intervalo 90 % (fuera de muestra)", 0.8 <= float(c) <= 0.97,
              valor=round(float(c), 3), umbral="[0.80, 0.97]", dimension="calibración")

ESQ_PRED = ("cod_dpto INT, anio INT, tasa_real DOUBLE, prediccion DOUBLE, tipo_prediccion STRING, tasa_anio_anterior DOUBLE, "
            "escenario STRING, modelo STRING, modelo_version STRING, limite_inferior_90 DOUBLE, limite_superior_90 DOUBLE, "
            "es_fuera_de_muestra BOOLEAN, nivel_prioridad STRING, orden_prioridad INT, tendencia_esperada STRING, error DOUBLE, "
            "error_absoluto DOUBLE, error_porcentual DOUBLE, dentro_intervalo_90 BOOLEAN, departamento STRING, region STRING, "
            "umbral_alta DOUBLE, umbral_critica DOUBLE")
cols_pred = [c.split()[0] for c in ESQ_PRED.split(", ")]
guardar_tabla(a_spark(preds[cols_pred], ESQ_PRED), SCHEMA_GOLD, "fact_predicciones",
              comentario="Predicciones fuera de muestra (backtest y prueba 2024), ajuste historico y pronostico 2025")

persistir_calidad()

✅ [modelado/nowcast] cobertura empírica del intervalo 90 % (fuera de muestra): valor=0.915 umbral=[0.80, 0.97] 
✅ [modelado/pronostico] cobertura empírica del intervalo 90 % (fuera de muestra): valor=0.924 umbral=[0.80, 0.97] 
  ✔ desercion_escolar.capa_oro.fact_predicciones (overwrite) → 561 filas
  ✔ desercion_escolar.gobierno_datos.control_calidad (append) → 190 filas


## 5. Resumen ejecutivo de la evaluación

In [ ]:
display(leer_tabla(SCHEMA_GOLD, "modelo_registro").filter(F.col("modelo_version") == MODELO_VERSION)
        .select("escenario", "familia", "rmse_validacion", "rmse_validacion_persistencia", "rmse_prueba",
                "rmse_prueba_ic95_inf", "rmse_prueba_ic95_sup", "rmse_prueba_persistencia", "mejora_vs_persistencia_pct",
                "r2_prueba", "supera_linea_base"))
display(leer_tabla(SCHEMA_GOLD, "modelo_metricas").filter("conjunto in ('validacion_global', 'prueba')").orderBy("escenario", "conjunto", "rmse"))
display(leer_tabla(SCHEMA_GOLD, "modelo_importancia").filter("ranking <= 15").orderBy("escenario", "ranking"))

escenario,familia,rmse_validacion,rmse_validacion_persistencia,rmse_prueba,rmse_prueba_ic95_inf,rmse_prueba_ic95_sup,rmse_prueba_persistencia,mejora_vs_persistencia_pct,r2_prueba,supera_linea_base
nowcast,RandomForest,1.3667711285722048,1.6819424196536288,0.5773803631144764,0.4719262024890241,0.6883047144768706,0.5293706955747864,-9.069196300630399,0.7852654941055828,false
pronostico,RandomForest,1.2792391788009818,1.7221964481616507,0.579016729710265,0.43722725330670675,0.7100006834891683,0.5293706955747864,-9.378311748362508,0.7840466015763543,false


escenario,modelo,conjunto,anio,n,mae,rmse,r2,sesgo
nowcast,Persistencia (t−1),prueba,2024,33,0.4015151515151515,0.5293706955747864,0.8194915191077818,0.19727272727272724
nowcast,RandomForest,prueba,2024,33,0.497071576435584,0.5773803631144764,0.7852654941055828,0.26887711298176875
nowcast,Media móvil 3 años,prueba,2024,33,0.4743434343434343,0.685425965519803,0.6973790964716746,0.3492929292929292
nowcast,GBT,prueba,2024,33,0.5698869203480728,0.7076581876461253,0.677429319560789,0.26467455607663193
nowcast,ElasticNet,prueba,2024,33,0.6587097620239195,0.7826653773547722,0.6054245472745414,0.05621260453377478
nowcast,Media móvil 3 años,validacion_global,null,132,1.1301767676767678,1.4595601765911912,0.29756007533523476,-0.42249999999999993
nowcast,RandomForest,validacion_global,null,132,1.117033016857123,1.460697548587922,0.2964648867977224,-0.42147137304402105
nowcast,ElasticNet,validacion_global,null,132,1.2004304312882144,1.5774529843219258,0.17950105858877718,-0.518189615145423
nowcast,Persistencia (t−1),validacion_global,null,132,1.2506060606060605,1.6819424196536288,0.06720238981340776,-0.17545454545454536
nowcast,GBT,validacion_global,null,132,1.2730984423910907,1.6880915197493271,0.0603693962091616,-0.4873285612287892


escenario,modelo,variable,importancia_permutacion,importancia_permutacion_std,importancia_nativa,tipo_importancia_nativa,ranking,fuente,grupo
nowcast,RandomForest,edu_tasa_desercion_media3,0.06656483119520326,0.06929752730740087,0.22504805503663175,featureImportances,1.0,MEN,Educativa (rezago)
nowcast,RandomForest,edu_tasa_desercion_lag1,0.04423843923547417,0.04618704887860134,0.14988962437961986,featureImportances,2.0,MEN,Educativa (rezago)
nowcast,RandomForest,edu_tasa_desercion_lag3,0.02849120022105168,0.03181921148366216,0.06198031873099733,featureImportances,3.0,MEN,Educativa (rezago)
nowcast,RandomForest,edu_desercion_primaria_lag1,0.025958677637007516,0.045761786770450175,0.0879038802863044,featureImportances,4.0,MEN,Educativa (rezago)
nowcast,RandomForest,edu_tasa_desercion_lag2,0.02047361156630525,0.03400512938565119,0.06405315127734874,featureImportances,5.0,MEN,Educativa (rezago)
nowcast,RandomForest,edu_desercion_secundaria_lag1,0.012122915636516934,0.033911978837150414,0.07762512687464235,featureImportances,6.0,MEN,Educativa (rezago)
nowcast,RandomForest,edu_desercion_media_lag1,0.009357953639755428,0.0176733266067263,0.06572395250697408,featureImportances,7.0,MEN,Educativa (rezago)
nowcast,RandomForest,pob_trabajo_informal,0.003113346221767636,0.00561926149381792,0.013212230095053444,featureImportances,8.0,Terridata (DNP),Socioeconómica
nowcast,RandomForest,pob_inasistencia_escolar,0.0028045730108723797,0.003569507076783366,0.02484703757550703,featureImportances,9.0,Terridata (DNP),Socioeconómica
nowcast,RandomForest,pob_trabajo_informal_en_cabecera,0.002777204496479069,0.007806326259480377,0.014839195314851427,featureImportances,10.0,Terridata (DNP),Socioeconómica


## 05_Validacion_Consumo_PowerBI

<!-- merged from: 05_Validacion_Consumo_PowerBI.ipynb (Jupy Tools) -->

# 05 · Validación de aceptación, vistas de consumo para Power BI y exportes

**CRISP-DM:** Evaluación + Despliegue · **Medallion:** Oro (capa de consumo)

1. Evalúa automáticamente la **lista mínima de aceptación** del informe (sección 10) y la persiste en
   `gobierno_datos.checklist_aceptacion`. Si un criterio crítico falla, el job termina en error.
2. Publica **vistas `vw_pbi_*`** estables (contrato de consumo para Power BI vía SQL Warehouse).
3. Opcional (`exportar_csv = si`): exporta las tablas de consumo a CSV en un Volume para Power BI Desktop sin conector.

In [ ]:
%run ./00_Config_Utilidades

Entorno Databricks: True | formato: delta | ejecución: 20261005T190944-e42c67


Utilidades cargadas ✔


In [ ]:
EXPORTAR_CSV = _widget("exportar_csv", "no").lower() in ("si", "sí", "true", "1", "yes")
checklist = []


def criterio(nombre, cumple, evidencia, critico=True):
    checklist.append((EJECUCION_ID, datetime.now(), nombre, bool(cumple), str(evidencia)[:1000], critico))
    print(f"{'✅' if cumple else ('❌' if critico else '⚠️')} {nombre}: {evidencia}")


def n_filas(schema, tabla):
    return leer_tabla(schema, tabla).count() if existe_tabla(schema, tabla) else -1

## 1. Lista mínima de aceptación

In [ ]:
TABLAS = {
    SCHEMA_BRONZE: ["sivigila", "pobreza", "educacion"],
    SCHEMA_SILVER: ["sivigila_limpio", "sivigila_cuarentena", "sivigila_departamento_anio", "sivigila_departamento_anio_perfil",
                    "pobreza_departamento_indicador", "pobreza_catalogo_indicadores", "educacion_departamento_anio",
                    "educacion_calidad_columnas"],
    SCHEMA_GOLD: ["dim_departamento", "dim_anio", "dim_evento_sivigila", "dim_perfil_epidemiologico", "dim_indicador_pobreza", "feature_store_desercion",
                  "diccionario_variables", "analisis_correlaciones", "analisis_interaccion", "fact_educacion",
                  "fact_sivigila_perfil", "fact_pobreza", "fact_predicciones", "modelo_metricas", "modelo_registro",
                  "modelo_importancia", "modelo_metricas_departamento"],
    SCHEMA_GOB: ["control_cargas", "control_calidad", "linaje_columnas", "perfil_bronce", "perfil_nulos_feature_store"],
}
conteos = {f"{s}.{t}": n_filas(s, t) for s, ts in TABLAS.items() for t in ts}
faltan = [k for k, v in conteos.items() if v <= 0]
criterio("Ejecución completa: todas las tablas de cada capa existen y tienen filas", not faltan,
         f"faltantes/vacías={faltan}" if faltan else f"{len(conteos)} tablas OK")
display(spark.createDataFrame([(k, v) for k, v in conteos.items()], "tabla STRING, filas BIGINT"))

for schema, tabla in [(SCHEMA_SILVER, "educacion_departamento_anio"), (SCHEMA_SILVER, "sivigila_departamento_anio"),
                      (SCHEMA_GOLD, "feature_store_desercion")]:
    dup = leer_tabla(schema, tabla).groupBy("cod_dpto", "anio").count().filter("count > 1").count()
    criterio(f"Una fila por departamento-año en {tabla}", dup == 0, f"duplicados={dup}")

fs = leer_tabla(SCHEMA_GOLD, "feature_store_desercion")
esperado = len(CODIGOS_DEPARTAMENTO) * (ANIO_PRONOSTICO - ANIO_MIN + 1)
criterio("Cardinalidad del join (33 × años, sin fan-out)", fs.count() == esperado, f"{fs.count()} vs {esperado}")
obj = fs.filter(F.col("anio").between(ANIO_MIN, ANIO_MAX) & F.col("tasa_desercion").isNotNull()).count()
criterio("Variable objetivo completa 2018-2024", obj == len(CODIGOS_DEPARTAMENTO) * (ANIO_MAX - ANIO_MIN + 1), f"{obj} observaciones")

criterio("% de nulos antes (Bronce) y después (Oro) documentado",
         conteos[f"{SCHEMA_GOB}.perfil_bronce"] > 0 and conteos[f"{SCHEMA_GOB}.perfil_nulos_feature_store"] > 0,
         "gobierno_datos.perfil_bronce y perfil_nulos_feature_store")

calidad = leer_tabla(SCHEMA_GOB, "control_calidad")
fallas = calidad.filter("estado = 'FALLA'").join(
    calidad.groupBy("capa").agg(F.max("fecha_evaluacion").alias("ult")), "capa").filter("fecha_evaluacion >= ult - INTERVAL 1 DAY").count()
criterio("Sin reglas de calidad bloqueantes incumplidas en la última ejecución", fallas == 0, f"fallas={fallas}")
alertas = calidad.filter("estado = 'ALERTA'").select("capa", "tabla", "regla", "valor_observado", "detalle").distinct()
print("Alertas vigentes (no bloqueantes, deben documentarse como limitaciones):")
display(alertas)

conserva = calidad.filter(F.col("regla").contains("conservación de registros")).orderBy(F.desc("fecha_evaluacion")).first()
criterio("SIVIGILA: todos los casos de Bronce quedan contabilizados (sin dropDuplicates)", conserva is not None and conserva.estado == "OK",
         conserva.detalle if conserva else "regla no encontrada")

met = leer_tabla(SCHEMA_GOLD, "modelo_metricas")
conjuntos = {r[0] for r in met.select("conjunto").distinct().collect()}
criterio("Métricas de validación y prueba", {"validacion", "validacion_global", "prueba"} <= conjuntos, sorted(conjuntos))
criterio("Línea base comparativa (persistencia)", met.filter(F.col("modelo").startswith("Persistencia")).count() > 0, "Persistencia y media móvil")
folds = met.filter("conjunto = 'validacion'").select("escenario", "anio").distinct().groupBy("escenario").count().collect()
criterio("Backtesting temporal con ≥ 3 pliegues por escenario", all(r["count"] >= 3 for r in folds) and len(folds) > 0,
         {r["escenario"]: r["count"] for r in folds})
md = leer_tabla(SCHEMA_GOLD, "modelo_metricas_departamento").groupBy("escenario").count().collect()
criterio("Métricas por departamento", all(r["count"] == len(CODIGOS_DEPARTAMENTO) for r in md) and len(md) > 0,
         {r["escenario"]: r["count"] for r in md})
reg = leer_tabla(SCHEMA_GOLD, "modelo_registro").orderBy(F.desc("fecha_entrenamiento")).limit(2).collect()
criterio("Registro de modelo (versión, variables, métricas, ruta, versión de datos)", len(reg) > 0,
         [(r.escenario, r.familia, r.modelo_version, round(r.rmse_prueba, 4)) for r in reg])
for r in reg:
    criterio(f"[{r.escenario}] el modelo supera la persistencia en validación y prueba", r.supera_linea_base,
             f"RMSE prueba {r.rmse_prueba:.4f} vs persistencia {r.rmse_prueba_persistencia:.4f} "
             f"(IC95 modelo [{r.rmse_prueba_ic95_inf:.4f}, {r.rmse_prueba_ic95_sup:.4f}])", critico=False)
dicc = leer_tabla(SCHEMA_GOLD, "diccionario_variables")
criterio("Disponibilidad temporal de cada variable documentada", dicc.filter(F.col("disponibilidad").isNull()).count() == 0,
         f"{dicc.count()} variables con fuente, disponibilidad y escenario")
criterio("Objetivo 4: perfiles infecciosos vs nutricionales e interacción con pobreza", conteos[f"{SCHEMA_GOLD}.analisis_interaccion"] > 0,
         "analisis_interaccion, analisis_cuadrantes, fact_sivigila_perfil")

guardar_tabla(spark.createDataFrame(checklist, "ejecucion_id STRING, fecha TIMESTAMP, criterio STRING, cumple BOOLEAN, evidencia STRING, critico BOOLEAN"),
              SCHEMA_GOB, "checklist_aceptacion", modo="append")

✅ Ejecución completa: todas las tablas de cada capa existen y tienen filas: 33 tablas OK


tabla,filas
capa_bronce.sivigila,3056235
capa_bronce.pobreza,83148
capa_bronce.educacion,462
capa_plata.sivigila_limpio,3047498
capa_plata.sivigila_cuarentena,8737
capa_plata.sivigila_departamento_anio,231
capa_plata.sivigila_departamento_anio_perfil,2673
capa_plata.pobreza_departamento_indicador,19776
capa_plata.pobreza_catalogo_indicadores,132
capa_plata.educacion_departamento_anio,462


✅ Una fila por departamento-año en educacion_departamento_anio: duplicados=0
✅ Una fila por departamento-año en sivigila_departamento_anio: duplicados=0
✅ Una fila por departamento-año en feature_store_desercion: duplicados=0
✅ Cardinalidad del join (33 × años, sin fan-out): 264 vs 264
✅ Variable objetivo completa 2018-2024: 231 observaciones
✅ % de nulos antes (Bronce) y después (Oro) documentado: gobierno_datos.perfil_bronce y perfil_nulos_feature_store
✅ Sin reglas de calidad bloqueantes incumplidas en la última ejecución: fallas=0
Alertas vigentes (no bloqueantes, deben documentarse como limitaciones):


capa,tabla,regla,valor_observado,detalle
plata,educacion,columnas no confiables (excluidas del modelado),7.0,"['aprobacion_transicion', 'aprobacion_primaria', 'aprobacion_secundaria', 'reprobacion_transicion', 'reprobacion_primaria', 'reprobacion_secundaria', 'repitencia_transicion']"
plata,sivigila,consistencia código DIVIPOLA ↔ nombre de departamento,161869.0,


✅ SIVIGILA: todos los casos de Bronce quedan contabilizados (sin dropDuplicates): 
✅ Métricas de validación y prueba: ['prueba', 'validacion', 'validacion_global']
✅ Línea base comparativa (persistencia): Persistencia y media móvil
✅ Backtesting temporal con ≥ 3 pliegues por escenario: {'nowcast': 4, 'pronostico': 3}
✅ Métricas por departamento: {'nowcast': 33, 'pronostico': 33}
✅ Registro de modelo (versión, variables, métricas, ruta, versión de datos): [('pronostico', 'RandomForest', 'v20261005162948', 0.579), ('nowcast', 'RandomForest', 'v20261005162948', 0.5774)]
⚠️ [pronostico] el modelo supera la persistencia en validación y prueba: RMSE prueba 0.5790 vs persistencia 0.5294 (IC95 modelo [0.4372, 0.7100])
⚠️ [nowcast] el modelo supera la persistencia en validación y prueba: RMSE prueba 0.5774 vs persistencia 0.5294 (IC95 modelo [0.4719, 0.6883])
✅ Disponibilidad temporal de cada variable documentada: 151 variables con fuente, disponibilidad y escenario
✅ Objetivo 4: perfiles infec

18

## 2. Vistas de consumo para Power BI (contrato estable)

In [ ]:
def vista(nombre, sql):
    nombre_full = fqtn(SCHEMA_GOLD, nombre)
    try:
        spark.sql(f"CREATE OR REPLACE VIEW {nombre_full} AS {sql}")
        print(f"  ✔ vista {nombre_full}")
    except Exception as e:
        print(f"  ✖ vista {nombre_full}: {str(e)[:200]}")


g = lambda t: fqtn(SCHEMA_GOLD, t)
vista("vw_pbi_predicciones_vigentes", f"""
    SELECT p.* FROM {g('fact_predicciones')} p
    WHERE p.modelo_version = (SELECT MAX(modelo_version) FROM {g('fact_predicciones')})""")
vista("vw_pbi_registro_vigente", f"""
    SELECT * FROM {g('modelo_registro')}
    WHERE modelo_version = (SELECT MAX(modelo_version) FROM {g('modelo_registro')})""")
vista("vw_pbi_resumen_departamental", f"""
    SELECT e.cod_dpto, d.departamento, d.region, d.latitud, d.longitud, d.nombre_mapa, e.anio,
           e.tasa_desercion, e.desercion_primaria, e.desercion_secundaria, e.desercion_media,
           e.tasa_desercion_nacional_ponderada, e.brecha_vs_nacional, e.poblacion_5_16,
           s.casos_total, s.casos_grupo_infecciosa, s.casos_grupo_nutricional, s.casos_grupo_psicosocial,
           s.casos_total / e.poblacion_5_16 * 1000 AS tasa_sivigila_por_1000_nna
    FROM {g('fact_educacion')} e
    JOIN {g('dim_departamento')} d ON d.cod_dpto = e.cod_dpto
    LEFT JOIN {fqtn(SCHEMA_SILVER, 'sivigila_departamento_anio')} s ON s.cod_dpto = e.cod_dpto AND s.anio = e.anio
    WHERE e.anio BETWEEN {ANIO_MIN} AND {ANIO_MAX}""")
vista("vw_pbi_calidad_ultima_ejecucion", f"""
    SELECT * FROM {fqtn(SCHEMA_GOB, 'control_calidad')}
    WHERE fecha_evaluacion >= (SELECT MAX(fecha_evaluacion) - INTERVAL 1 DAY FROM {fqtn(SCHEMA_GOB, 'control_calidad')})""")

  ✔ vista desercion_escolar.capa_oro.vw_pbi_predicciones_vigentes
  ✔ vista desercion_escolar.capa_oro.vw_pbi_registro_vigente
  ✔ vista desercion_escolar.capa_oro.vw_pbi_resumen_departamental
  ✔ vista desercion_escolar.capa_oro.vw_pbi_calidad_ultima_ejecucion


## 3. Exportes CSV opcionales (Power BI Desktop sin conector Databricks)

In [ ]:
TABLAS_PBI = ["dim_departamento", "dim_anio", "dim_evento_sivigila", "dim_perfil_epidemiologico", "dim_indicador_pobreza", "fact_educacion",
              "fact_sivigila_perfil", "fact_pobreza", "fact_predicciones", "modelo_metricas", "modelo_registro",
              "modelo_importancia", "modelo_metricas_departamento", "modelo_metricas_subgrupo", "analisis_correlaciones",
              "analisis_interaccion", "analisis_cuadrantes", "analisis_picos_desercion", "clasificacion_territorial",
              "diccionario_variables"]
if EXPORTAR_CSV:
    for t in TABLAS_PBI:
        if existe_tabla(SCHEMA_GOLD, t):
            df = leer_tabla(SCHEMA_GOLD, t)
            df = df.select(*[F.to_json(F.col(c)).alias(c) if str(df.schema[c].dataType).startswith("ArrayType") else F.col(c) for c in df.columns])
            (df.coalesce(1).write.mode("overwrite").option("header", True).option("encoding", "UTF-8")
             .csv(f"{EXPORT_PATH}/{t}"))
            print(f"  ✔ {EXPORT_PATH}/{t}/part-*.csv")
    for t in ["control_calidad", "checklist_aceptacion", "control_cargas"]:
        (leer_tabla(SCHEMA_GOB, t).coalesce(1).write.mode("overwrite").option("header", True).csv(f"{EXPORT_PATH}/{t}"))
        print(f"  ✔ {EXPORT_PATH}/{t}/part-*.csv")
else:
    print("Exportación CSV desactivada (exportar_csv = no). Power BI se conecta a las tablas/vistas Oro vía SQL Warehouse.")

Exportación CSV desactivada (exportar_csv = no). Power BI se conecta a las tablas/vistas Oro vía SQL Warehouse.


In [ ]:
criticos_fallidos = [c[2] for c in checklist if c[5] and not c[3]]
persistir_calidad()
if criticos_fallidos:
    raise ValueError(f"Criterios críticos de aceptación incumplidos: {criticos_fallidos}")
print("✅ Pipeline aceptado: todos los criterios críticos se cumplen.")

✅ Pipeline aceptado: todos los criterios críticos se cumplen.
